# PhilGEPS Public Bidding: Data Preparation, Exploratory Analysis and Procurement-Failure Risk Modeling

**LT8 | Machine Learning Project | Documented analytical notebook (original run)**
**Data source of this run:** Quarterly PhilGEPS extracts, **2023 Q1 to 2026 Q2** (14 files; 2026 taken from the PhilGEPS 2.0 exports only).
**Scope:** Public Bidding and the 2026 equivalent, *Competitive Bidding (Public Bidding)*.

> **Continued in `notebooks/`.** The full analytical roadmap below has been re-run on the expanded, merged dataset: PhilGEPS 1.5 for 2023 Q1–2026 Q3 plus PhilGEPS 2.0 for 2026 Q1–Q3, 18 files combined by `scripts/combine_raw.py`. See `notebooks/01_data_preparation_and_eda.ipynb` (steps 1–5), `notebooks/02_modeling.ipynb` (step 6) and `notebooks/03_diagnostics_and_interpretation.ipynb` (steps 7–9). This notebook is kept unchanged as the record of the original 14-file run. Its saved outputs and figures refer to that run.

## Project purpose and business question

**Business question:** Using only information available when a public procurement notice is published, how likely is it that the procurement **fails**, ending in a `Failed` bidding rather than an award?

The deliverable is a **procurement-failure risk score** for notices. Its intended use is risk screening and review prioritisation: an agency, oversight body or analyst can use the ranking to decide which notices warrant earlier scrutiny and preventive action. **It is not a model of whether any individual supplier will win a contract, and it does not recommend which bids a supplier should pursue.** Whether a notice is awarded says nothing about a specific supplier's chance of winning it.

The notebook documents both the investigation that led to a usable dataset and the subsequent modeling work. It intentionally retains the original code, saved outputs, checks and diagnostic experiments.

## Reviewer notes and where they are addressed

The reviewer suggested positioning the project as **predicting the likelihood of procurement failure** rather than recommending which bids suppliers should pursue. This is adopted throughout. The reviewer's four questions:

| # | Reviewer question | Where it is addressed | Answer |
|---|---|---|---|
| 1 | How does predicting whether a bid is awarded or failed help suppliers decide which bids to pursue? An awarded bid does not mean a particular supplier has a good chance of winning. | This header; Phase 9; `notebooks/03` §9 | Agreed. The project now predicts **procurement-failure risk** for a notice. It serves agencies and oversight first, for prevention and review prioritisation. For a supplier it is only a signal that preparing a bid for a notice may be wasted effort. It is not a win-probability model. |
| 2 | Are all proposed features available when the bid notice is first published? | §6.1; `notebooks/01` §5.6 (availability of every field) | Yes. All predictors are part of the notice when it is posted: approved budget, funding source and instrument, procurement category, classification, region, organisation type, publication date, closing date (bidding window), contract duration and number of line items. Award and contract fields (award date, contract amount, awardee, award notice status) are never used. Remaining caveat: the extracts are final-state snapshots, so later edits to a notice cannot be detected. |
| 3 | How are ongoing, cancelled or closed bids handled? They should not automatically be treated as failed. | §2.2, §5.2, §7.5; `notebooks/01` §5.2; `notebooks/02` §6.10 | They are **not** treated as failed. A bid is labelled only when all its line items are `Awarded` (0) or all are `Failed` (1). `Active`, `Closed`, `Cancelled`, `Shortlisted` and conflicting bids stay unlabelled and are excluded. Counting cancellations as failures is tested separately as a sensitivity analysis. Label coverage by period is reported, because recent notices are often still unresolved. |
| 4 | How many observations are there, and what is the awarded/failed distribution? | §5.7–5.8 (original run); `notebooks/01` §5.7 (current data) | Current merged data: **1,277,213** Public Bidding bids after quality exclusions. **573,585** have a final outcome: **504,350 Awarded** and **69,235 Failed** (12.1% failed, about 1 failure per 7.3 awards). 703,628 bids are unlabelled. Original run: 501,298 modelling bids (441,738 Awarded / 59,733 Failed before exclusions). |

## Analytical roadmap

1. **Source preparation:** Locate, convert and inventory 14 quarterly files.
2. **Initial EDA:** Inspect the source schema, isolate public bidding, and study status labels and repeated bid references.
3. **Data-quality investigation:** Identify and repair the shifted 2025 Q4 columns; harmonize 2026 procurement terminology.
4. **Consolidation:** Assemble item-level public-bidding records across quarters and audit coverage.
5. **Bid-level modeling table:** Resolve one outcome per bid reference; audit features and exclude ambiguous records.
6. **Modeling:** Train on 2023–2024, validate on 2025, and compare three classifiers.
7. **Out-of-time diagnostics:** Examine 2026 performance, reporting changes, label coverage, and feature importance.
8. **Business interpretation:** Evaluate a fixed-capacity review queue and explore budget/category relationships without claiming causality.

### Reading and execution notes

- **The notebook is a documented research trail, not a one-click `Run All` pipeline.** Some earlier cells are exploratory, some depend on previous files or variables, and the original notebook includes a historical cleanup cell that **deletes a generated 2025 Q4 CSV**. Read the notes before rerunning any code.
- The saved notebook outputs record prior executions. **No dataset files are embedded in the notebook**; regeneration requires the original `dataset/` directory, sufficient disk space, and the Python dependencies.
- **The raw files have been renamed since this run** (now `1.5_YYYY_QN.csv` and `2.0_YYYY_QN.csv` in `data/raw/`); the mapping is in §1.1. For a reproducible run on the repository data, use the notebooks in `notebooks/`.
- Files are written relative to the Jupyter working folder, primarily into `dataset_csv/` and `dataset_cleaned/`.
- The later **consolidated** and **bid-level** steps supersede the preliminary, single-quarter EDA. Their final counts are the ones to use when reporting this run.
- The 2026 data in this run ends at **2026 Q2** and comes from the PhilGEPS 2.0 exports only. The repository now also holds the PhilGEPS 1.5 exports for 2026 Q1–Q3, which carry most of the 2026 volume; they are used in `notebooks/`.

## Phase 1. Source preparation and reproducibility

**Purpose:** Establish where Jupyter is running so the source and output folders can be found. The original execution uses a working folder named `ML_Govt_Bidding`.

In [1]:

from pathlib import Path

print("Jupyter working folder:", Path.cwd())


Jupyter working folder: C:\Users\User\OneDrive - ASIAN INSTITUTE OF MANAGEMENT\Documents\ML_Govt_Bidding\ML_Govt_Bidding


### 1.1 Convert and standardize quarterly extracts

The next cell maps the 14 original quarterly filenames to a consistent `PHILGEPS_YYYY_Q#.csv` naming convention. It checks common text encodings, detects delimiters, and writes UTF-8 CSV outputs in `dataset_csv/`. Existing output files are skipped; missing sources are reported.

**Why:** Consistent filenames and encoding allow the same processing logic to handle multiple years of data. Because the files are large, subsequent steps read them in chunks.

**Filename update.** The raw files in `data/raw/` have been renamed since this cell was written, and four files have been added. The mapping below uses the original download names:

| Original download name (used below) | Current repository name |
|---|---|
| `2023_(JAN-MAR).csv` … `2025_(JUL-SEPT).csv` (11 files) | `1.5_2023_Q1.csv` … `1.5_2025_Q3.csv` |
| `2025_Oct-Dec.csv` | `1.5_2025_Q4.csv` |
| `DATASETS_MPHILGEPS_2026_JAN-MAR.csv`, `..._APR-JUNE.csv` | `2.0_2026_Q1.csv`, `2.0_2026_Q2.csv` |
| *(not in this run)* | `1.5_2026_Q1.csv` … `1.5_2026_Q3.csv`, `2.0_2026_Q3.csv` |

The conversion, delimiter detection and 2025 Q4 column-shift repair are now implemented in `scripts/combine_raw.py`, which writes `data/processed/philgeps_combined.parquet`.


In [2]:

from pathlib import Path
import csv
import codecs

# 1. Locate dataset folder
base = Path.cwd()

if base.name.lower() == "dataset":
    source_dir = base
else:
    source_dir = base / "dataset"

if not source_dir.is_dir():
    raise FileNotFoundError(
        f"Dataset folder not found: {source_dir}"
    )

output_dir = source_dir.parent / "dataset_csv"
output_dir.mkdir(exist_ok=True)

# 2. Standardized filenames
file_map = {
    "2023_(JAN-MAR)": "PHILGEPS_2023_Q1.csv",
    "2023_(APR-JUN)": "PHILGEPS_2023_Q2.csv",
    "2023_(JUL-SEPT)": "PHILGEPS_2023_Q3.csv",
    "2023_(OCT-DEC)": "PHILGEPS_2023_Q4.csv",
    "2024_Jan-Mar": "PHILGEPS_2024_Q1.csv",
    "2024_Apr-Jun": "PHILGEPS_2024_Q2.csv",
    "2024_Jul-Sep": "PHILGEPS_2024_Q3.csv",
    "2024_Oct-Dec": "PHILGEPS_2024_Q4.csv",
    "2025_(JAN-MAR)": "PHILGEPS_2025_Q1.csv",
    "2025_(APR-JUN)": "PHILGEPS_2025_Q2.csv",
    "2025_(JUL-SEPT)": "PHILGEPS_2025_Q3.csv",
    "2025_Oct-Dec": "PHILGEPS_2025_Q4.csv",
    "DATASETS_MPHILGEPS_2026_JAN-MAR":
        "PHILGEPS_2026_Q1.csv",
    "DATASETS_MPHILGEPS_2026_APR-JUNE":
        "PHILGEPS_2026_Q2.csv",
}

# 3. Identify character encoding
def get_encoding(path):
    with open(path, "rb") as f:
        sample = f.read(65536)

    if sample.startswith((
        codecs.BOM_UTF16_LE,
        codecs.BOM_UTF16_BE
    )):
        return "utf-16"

    try:
        sample.decode("utf-8-sig")
        return "utf-8-sig"
    except UnicodeDecodeError:
        return "cp1252"

# 4. Convert each file
available = {
    p.stem.casefold(): p
    for p in source_dir.iterdir()
    if p.is_file()
}

csv.field_size_limit(10_000_000)

for old_name, new_name in file_map.items():

    source = available.get(old_name.casefold())

    if source is None:
        print(f"NOT FOUND: {old_name}")
        continue

    destination = output_dir / new_name

    if destination.exists():
        print(f"ALREADY EXISTS: {new_name}")
        continue

    encoding = get_encoding(source)

    with open(
        source, "r",
        encoding=encoding,
        errors="replace",
        newline=""
    ) as f:
        sample = f.read(65536)

    try:
        dialect = csv.Sniffer().sniff(
            sample, delimiters=",\t;|"
        )
        delimiter = dialect.delimiter
    except csv.Error:
        print(f"CHECK DELIMITER: {old_name}")
        continue

    with open(
        source, "r",
        encoding=encoding,
        errors="replace",
        newline=""
    ) as infile, open(
        destination, "w",
        encoding="utf-8-sig",
        newline=""
    ) as outfile:

        reader = csv.reader(
            infile, delimiter=delimiter
        )
        writer = csv.writer(outfile)

        header = next(reader, None)

        if not header or len(header) < 2:
            destination.unlink()
            print(f"CHECK FORMAT: {old_name}")
            continue

        writer.writerow(header)
        rows = 0

        for record in reader:
            writer.writerow(record)
            rows += 1

    print(f"CREATED: {new_name} | {rows:,} data rows")

print("\nConversion process finished.")
print("Output:", output_dir)


ALREADY EXISTS: PHILGEPS_2023_Q1.csv
ALREADY EXISTS: PHILGEPS_2023_Q2.csv
ALREADY EXISTS: PHILGEPS_2023_Q3.csv
ALREADY EXISTS: PHILGEPS_2023_Q4.csv
ALREADY EXISTS: PHILGEPS_2024_Q1.csv
ALREADY EXISTS: PHILGEPS_2024_Q2.csv
ALREADY EXISTS: PHILGEPS_2024_Q3.csv
ALREADY EXISTS: PHILGEPS_2024_Q4.csv
ALREADY EXISTS: PHILGEPS_2025_Q1.csv
ALREADY EXISTS: PHILGEPS_2025_Q2.csv
ALREADY EXISTS: PHILGEPS_2025_Q3.csv
ALREADY EXISTS: PHILGEPS_2025_Q4.csv
ALREADY EXISTS: PHILGEPS_2026_Q1.csv
ALREADY EXISTS: PHILGEPS_2026_Q2.csv

Conversion process finished.
Output: C:\Users\User\OneDrive - ASIAN INSTITUTE OF MANAGEMENT\Documents\ML_Govt_Bidding\ML_Govt_Bidding\dataset_csv


### 1.2 Historical cleanup of a partial conversion **(not a routine step)**

**Important:** This original cell deletes `dataset_csv/PHILGEPS_2025_Q4.csv` if present. It was used to remove an incomplete file during an earlier attempt. **Do not execute it during an ordinary rerun.** The conversion step above must be rerun after any deletion. This housekeeping operation is retained only to document what happened.

In [3]:

# Remove the incomplete output from the failed attempt

partial_file = output_dir / "PHILGEPS_2025_Q4.csv"

if partial_file.exists():
    partial_file.unlink()
    print("Incomplete file deleted.")
else:
    print("No incomplete file found.")


Incomplete file deleted.


### 1.3 Check a small source sample and column names

A five-row sample from **2025 Q3** is inspected to see the raw data structure before reading entire files. Source fields include procurement entity, bid reference, notice details, budget, publication date, item details, and bid/award statuses.

In [4]:

import pandas as pd

file_path = output_dir / "PHILGEPS_2025_Q3.csv"

df = pd.read_csv(file_path, nrows=5)

display(df)
print("Columns:", df.columns.tolist())


,Procuring Entity (PE),Region,Province,City/Municipality,Government Branch,PE Organization Type,PE Organization Type (Grouped),Bid Reference No.,Notice Title,Classification,...,Notice to Proceed Date,Contract Effectivity Date,Contract End Date,Awardee Organization Name,Country of Awardee,Region of Awardee,Province of Awardee,City/Municipality of Awardee,Awardee Size,Awardee Joint Venture
0,DOJ-NATIONAL PROSECUTION SERVICE XI,Region XI,Davao Del Sur,Davao City,Executive,Department,National Government Agencies (NGA),0,Venue with Meals for the 2025 TEPLEA on 12 Sep...,Goods - General Support Services,...,08/08/2025,08/08/2025,12/09/2025,FABIAN'S CATERING SERVICES,Philippines,Region XI,Davao Del Sur,Digos City,Micro,NaN
1,"CITY OF KORONADAL, SOUTH COTABATO",Region XII,South Cotabato,Koronadal City,Executive,City Government,Local Government Units (LGU),0,LEASE OF VENUE & ACCOMMODATION W/ MEALS & SNAC...,Goods,...,25/06/2025,04/07/2025,06/07/2025,ISLA JARDIN DEL MAR,Philippines,Region XII,Sarangani,Glan,Medium,NaN
2,SOCIAL SECURITY SYSTEM - LUZON CENTRAL 2,Region III,Pampanga,San Fernando City,Executive,Government Owned and/or Controlled Corp.(GOCC),Government Owned and Controlled Corporations (...,0,PROCUREMENT OF MEALS (AM SNACK AND LUNCH) AND ...,Civil Works,...,NaN,NaN,NaN,MALOLOS RESORT CLUB ROYALE INC.,Philippines,Region III,Bulacan,Malolos City,Medium,NaN
3,"MUNICIPALITY OF PAGSANJAN, LAGUNA",Region IV-A,Laguna,Pagsanjan,Executive,Municipal Government,Local Government Units (LGU),0,AUDIO-VISUAL SYSTEM FOR THE INAUGURATION OF EL...,Goods,...,NaN,NaN,NaN,PAGE068 LIGHTS AND SOUNDS RENTAL,Philippines,Region IV-A,Laguna,Sta. Cruz,Micro,NaN
4,"MUNICIPALITY OF MABINI, PANGASINAN",Region I,Pangasinan,Mabini,Executive,Municipal Government,Local Government Units (LGU),0,Purchase of Diesel Fuel for the Office of the ...,Goods,...,10/07/2025,11/07/2025,20/07/2025,JAKARAM CORP.,Philippines,Region I,Pangasinan,Mabini,Medium,NaN


Columns: ['Procuring Entity (PE)', 'Region', 'Province', 'City/Municipality', 'Government Branch', 'PE Organization Type', 'PE Organization Type (Grouped)', 'Bid Reference No.', 'Notice Title', 'Classification', 'Procurement Mode', 'Business Category', 'Funding Source', 'Funding Instrument', 'Trade Agreement', 'Approved Budget of the Contract', 'Published Date', 'Closing Date', 'Area of Delivery', 'Contract Duration', 'Calendar Type', 'Line Item No', 'Item Name', 'Item Description', 'Quantity', 'UOM', 'Item Budget', 'Bid Notice Status', 'Award Reference No.', 'Award Title', 'UNSPSC Code', 'UNSPSC Description', 'Published Date(Award)', 'Award Date', 'Contract Amount', 'Award Notice Status', 'Notice to Proceed Date', 'Contract Effectivity Date', 'Contract End Date', 'Awardee Organization Name', 'Country of Awardee', 'Region of Awardee', 'Province of Awardee', 'City/Municipality of Awardee', 'Awardee Size', 'Awardee Joint Venture']


### 1.4 Inventory files and compare schemas

The inventory confirms the expected 14 quarter-level CSVs. A header comparison checks whether the field layout is consistent. This first check finds **46 fields** for 2023–2025 and **45 fields** for the 2026 extracts; later diagnostics also identify a *within-row alignment problem* in 2025 Q4 that a header count alone cannot catch.

In [5]:

from pathlib import Path
import pandas as pd

files = sorted(output_dir.glob("PHILGEPS_*.csv"))

print("Total CSV files:", len(files))

for file in files:
    print(file.name)

if len(files) == 14:
    print("\nAll 14 quarterly files are available.")
else:
    print("\nExpected 14 files. Check missing files.")


Total CSV files: 14
PHILGEPS_2023_Q1.csv
PHILGEPS_2023_Q2.csv
PHILGEPS_2023_Q3.csv
PHILGEPS_2023_Q4.csv
PHILGEPS_2024_Q1.csv
PHILGEPS_2024_Q2.csv
PHILGEPS_2024_Q3.csv
PHILGEPS_2024_Q4.csv
PHILGEPS_2025_Q1.csv
PHILGEPS_2025_Q2.csv
PHILGEPS_2025_Q3.csv
PHILGEPS_2025_Q3_PUBLIC.csv
PHILGEPS_2026_Q1.csv
PHILGEPS_2026_Q2.csv

All 14 quarterly files are available.


In [6]:

schema_report = []
reference_columns = None

for file in files:
    df_sample = pd.read_csv(
        file,
        nrows=5,
        encoding="utf-8-sig"
    )

    columns = df_sample.columns.tolist()

    if reference_columns is None:
        reference_columns = columns

    schema_report.append({
        "File": file.name,
        "Number of Columns": len(columns),
        "Matches Reference": columns == reference_columns
    })

schema_df = pd.DataFrame(schema_report)
display(schema_df)



,File,Number of Columns,Matches Reference
0,PHILGEPS_2023_Q1.csv,46,True
1,PHILGEPS_2023_Q2.csv,46,True
2,PHILGEPS_2023_Q3.csv,46,True
3,PHILGEPS_2023_Q4.csv,46,True
4,PHILGEPS_2024_Q1.csv,46,True
5,PHILGEPS_2024_Q2.csv,46,True
6,PHILGEPS_2024_Q3.csv,46,True
7,PHILGEPS_2024_Q4.csv,46,True
8,PHILGEPS_2025_Q1.csv,46,True
9,PHILGEPS_2025_Q2.csv,46,True


### 1.5 Explore source fields and the first five rows

These quick probes list columns and inspect preliminary distributions. **These are sample-level checks, not representative population estimates.** In particular, the following exact Public Bidding filter initially returns zero because the `df` variable only holds five rows. The chunked scan that follows corrects this sampling limitation.

In [7]:

print("Available columns:\n")

for col in df.columns:
    print(col)


Available columns:

Procuring Entity (PE)
Region
Province
City/Municipality
Government Branch
PE Organization Type
PE Organization Type (Grouped)
Bid Reference No.
Notice Title
Classification
Procurement Mode
Business Category
Funding Source
Funding Instrument
Trade Agreement
Approved Budget of the Contract
Published Date
Closing Date
Area of Delivery
Contract Duration
Calendar Type
Line Item No
Item Name
Item Description
Quantity
UOM
Item Budget
Bid Notice Status
Award Reference No.
Award Title
UNSPSC Code
UNSPSC Description
Published Date(Award)
Award Date
Contract Amount
Award Notice Status
Notice to Proceed Date
Contract Effectivity Date
Contract End Date
Awardee Organization Name
Country of Awardee
Region of Awardee
Province of Awardee
City/Municipality of Awardee
Awardee Size
Awardee Joint Venture


In [8]:

print(df.columns.tolist())


['Procuring Entity (PE)', 'Region', 'Province', 'City/Municipality', 'Government Branch', 'PE Organization Type', 'PE Organization Type (Grouped)', 'Bid Reference No.', 'Notice Title', 'Classification', 'Procurement Mode', 'Business Category', 'Funding Source', 'Funding Instrument', 'Trade Agreement', 'Approved Budget of the Contract', 'Published Date', 'Closing Date', 'Area of Delivery', 'Contract Duration', 'Calendar Type', 'Line Item No', 'Item Name', 'Item Description', 'Quantity', 'UOM', 'Item Budget', 'Bid Notice Status', 'Award Reference No.', 'Award Title', 'UNSPSC Code', 'UNSPSC Description', 'Published Date(Award)', 'Award Date', 'Contract Amount', 'Award Notice Status', 'Notice to Proceed Date', 'Contract Effectivity Date', 'Contract End Date', 'Awardee Organization Name', 'Country of Awardee', 'Region of Awardee', 'Province of Awardee', 'City/Municipality of Awardee', 'Awardee Size', 'Awardee Joint Venture']


In [9]:

print(df["Funding Source"].value_counts(dropna=False))
print(df["Procurement Mode"].value_counts(dropna=False))


Funding Source
Government of the Philippines (GOP)    5
Name: count, dtype: int64
Procurement Mode
Negotiated Procurement - Lease of Real Property (Sec. 53.10)      3
Negotiated Procurement - Emergency Cases (Sec. 53.2)              1
Direct Retail Purchase of POL and Airline Tickets (Sec. 53.14)    1
Name: count, dtype: int64


In [10]:

df_public = df[
    df["Procurement Mode"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("public bidding")
].copy()

print("Public Bidding records:", len(df_public))


Public Bidding records: 0


In [11]:
df = pd.read_csv(file_path, nrows=5)

### 1.6 Scan the full 2025 Q3 file in chunks

Instead of drawing conclusions from five records, this step counts procurement methods across all **361,502** Q3 rows. It finds **107,856** Public Bidding item-level rows, which justifies focusing on that procurement method for the project.

In [12]:

import pandas as pd

file_path = output_dir / "PHILGEPS_2025_Q3.csv"

total_records = 0
public_bidding_records = 0
procurement_counts = {}

for chunk in pd.read_csv(
    file_path,
    chunksize=50000,
    encoding="utf-8-sig",
    low_memory=False
):
    total_records += len(chunk)

    modes = chunk["Procurement Mode"].astype("string").str.strip()

    counts = modes.value_counts(dropna=False)

    for mode, count in counts.items():
        key = str(mode)
        procurement_counts[key] = (
            procurement_counts.get(key, 0) + count
        )

    public_bidding_records += modes.str.contains(
        "Public Bidding",
        case=False,
        na=False,
        regex=False
    ).sum()

print("Total records:", f"{total_records:,}")
print("Public Bidding records:", f"{public_bidding_records:,}")

print("\nProcurement mode distribution:")
display(
    pd.Series(procurement_counts)
    .sort_values(ascending=False)
)


Total records: 361,502
Public Bidding records: 107,856

Procurement mode distribution:


Negotiated Procurement - Small Value Procurement (Sec. 53.9)                        188915
Public Bidding                                                                      107856
Shopping - Ordinary/Regular Office Supplies & Equipment (Sec. 52.1.b)                47492
Negotiated Procurement - Two Failed Biddings (Sec. 53.1)                              4630
Negotiated Procurement - Lease of Real Property (Sec. 53.10)                          4236
Negotiated Procurement - Community Participation (Sec. 53.12)                         2222
Direct Contracting (Sec. 50)                                                          1857
Shopping - Unforeseen Contingency (Sec. 52.1.a)                                       1181
Negotiated Procurement - Emergency Cases (Sec. 53.2)                                   909
Direct Retail Purchase of POL and Airline Tickets (Sec. 53.14)                         552
NP - Scientific, Scholarly/Artistic Work, Exclusive Tech. & Media Service (53.6)       431

## Phase 2. Initial Public Bidding EDA (2025 Q3 pilot)

**Purpose:** Create a manageable single-quarter pilot and test whether `Awarded` versus `Failed` is a meaningful label.

The pilot extracts **Public Bidding only** into `PHILGEPS_2025_Q3_PUBLIC.csv`; the subsequent investigations work on that filtered copy.

In [13]:

import pandas as pd

source_file = output_dir / "PHILGEPS_2025_Q3.csv"
filtered_file = output_dir / "PHILGEPS_2025_Q3_PUBLIC.csv"

first_chunk = True
total_public = 0

for chunk in pd.read_csv(
    source_file,
    chunksize=50000,
    encoding="utf-8-sig",
    low_memory=False
):
    mode = (
        chunk["Procurement Mode"]
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    filtered = chunk.loc[
        mode.eq("public bidding").fillna(False)
    ].copy()

    filtered.to_csv(
        filtered_file,
        mode="w" if first_chunk else "a",
        index=False,
        header=first_chunk,
        encoding="utf-8-sig"
    )

    first_chunk = False
    total_public += len(filtered)

print(f"Public Bidding records saved: {total_public:,}")
print("Saved to:", filtered_file)


Public Bidding records saved: 107,856
Saved to: C:\Users\User\OneDrive - ASIAN INSTITUTE OF MANAGEMENT\Documents\ML_Govt_Bidding\ML_Govt_Bidding\dataset_csv\PHILGEPS_2025_Q3_PUBLIC.csv


### 2.1 Understand the available outcome statuses

`Bid Notice Status` contains statuses such as **Closed, Awarded, Active, Cancelled, Failed,** and **Shortlisted**. `Award Notice Status` instead contains entries such as **Posted, Updated,** and **Cancelled**; it is not interchangeable with the bid outcome.

The next table cross-tabulates the two status fields to identify potentially inconsistent cases and unresolved notices.

In [14]:

public_df = pd.read_csv(
    filtered_file,
    low_memory=False
)

for col in ["Bid Notice Status", "Award Notice Status"]:
    if col in public_df.columns:
        print(f"\n{col}")
        display(public_df[col].value_counts(dropna=False))



Bid Notice Status


Bid Notice Status
Closed         69466
Awarded        16365
Active         14781
Cancelled       4619
Failed          2577
Shortlisted       48
Name: count, dtype: int64


Award Notice Status


Award Notice Status
NaN          89910
Updated      11836
Posted        6016
Cancelled       94
Name: count, dtype: int64

In [15]:

status_matrix = pd.crosstab(
    public_df["Bid Notice Status"].fillna("Missing"),
    public_df["Award Notice Status"].fillna("Missing"),
    margins=True
)

display(status_matrix)


Award Notice Status,Cancelled,Missing,Posted,Updated,All
Bid Notice Status,,,,,
Active,0,14781,0,0,14781
Awarded,93,1,4832,11439,16365
Cancelled,0,4619,0,0,4619
Closed,1,67884,1184,397,69466
Failed,0,2577,0,0,2577
Shortlisted,0,48,0,0,48
All,94,89910,6016,11836,107856


### 2.2 Prototype outcome labels

The first provisional rule keeps *explicitly* `Awarded` or `Failed` rows and assigns `Target_Success = 1` for Awarded, `0` for Failed. All other statuses are left **unlabeled**, not automatically counted as failures.

This sample is still **item-level**: one bid can have several item rows, so the counts are unsuitable for model training until deduplicated by reference.

In [16]:

# Normalize status descriptions
status = (
    public_df["Bid Notice Status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Retain explicitly labeled awarded or failed records
ml_df = public_df.loc[
    status.isin(["awarded", "failed"])
].copy()

ml_df["Target_Success"] = (
    status.loc[ml_df.index]
    .map({"awarded": 1, "failed": 0})
    .astype(int)
)

print("Initial labeled sample:", len(ml_df))
display(ml_df["Target_Success"].value_counts())


Initial labeled sample: 18942


Target_Success
1    16365
0     2577
Name: count, dtype: int64

### 2.3 Investigate disagreements between notice fields

Some records marked **Closed** already have a posted or updated award notice, and some marked **Awarded** have missing/cancelled award notices. These exceptions motivate a more careful, conservative outcome rule and a bid-reference-level audit; they should not be silently relabeled based on a single field.

In [17]:

# Normalize both status columns
bid_status = (
    public_df["Bid Notice Status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

award_status = (
    public_df["Award Notice Status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Closed records with an award notice
closed_with_award = public_df[
    bid_status.eq("closed") &
    award_status.isin(["posted", "updated"])
].copy()

# Awarded records with missing or cancelled award notice
awarded_to_review = public_df[
    bid_status.eq("awarded") &
    ~award_status.isin(["posted", "updated"])
].copy()

print("Closed with award notice:", len(closed_with_award))
print("Awarded requiring review:", len(awarded_to_review))

# Display relevant fields
review_columns = [
    "Bid Reference No.",
    "Bid Notice Status",
    "Award Notice Status",
    "Awardee Organisation Name",
    "Award Date",
    "Award Size"
]

review_columns = [
    c for c in review_columns
    if c in public_df.columns
]

display(closed_with_award[review_columns].head(10))
display(awarded_to_review[review_columns].head(10))


Closed with award notice: 1581
Awarded requiring review: 94


,Bid Reference No.,Bid Notice Status,Award Notice Status,Award Date
958,12173516,Closed,Updated,12/08/2025
959,12173516,Closed,Updated,12/08/2025
960,12173516,Closed,Updated,12/08/2025
961,12173516,Closed,Updated,12/08/2025
962,12173516,Closed,Updated,12/08/2025
1565,12176739,Closed,Updated,14/08/2025
1566,12176739,Closed,Updated,14/08/2025
1567,12176739,Closed,Updated,14/08/2025
1568,12176739,Closed,Updated,14/08/2025
1569,12176739,Closed,Updated,14/08/2025


,Bid Reference No.,Bid Notice Status,Award Notice Status,Award Date
662,12172392,Awarded,Cancelled,15/08/2025
664,12172416,Awarded,Cancelled,15/08/2025
1820,12177138,Awarded,NaN,NaN
2362,12178516,Awarded,Cancelled,15/08/2025
2364,12178516,Awarded,Cancelled,15/08/2025
2365,12178516,Awarded,Cancelled,15/08/2025
2367,12178516,Awarded,Cancelled,15/08/2025
2368,12178516,Awarded,Cancelled,15/08/2025
2369,12178516,Awarded,Cancelled,15/08/2025
2370,12178516,Awarded,Cancelled,15/08/2025


### 2.4 Check the unit of observation

One `Bid Reference No.` can appear across hundreds of item rows. An item-level model would disproportionately weight large multi-item procurements and could leak nearly identical records between training and validation. The next checks distinguish true duplicate rows from legitimate multiple line items and look for conflicting outcomes.

In [18]:

# Check duplicate procurement references

id_col = "Bid Reference No."

counts = (
    public_df[id_col]
    .astype("string")
    .str.strip()
    .value_counts()
)

print("Total records:", f"{len(public_df):,}")
print("Unique Bid References:", f"{len(counts):,}")
print("References appearing more than once:",
      f"{(counts > 1).sum():,}")

print("Records linked to repeated references:",
      f"{counts[counts > 1].sum():,}")

print("\nTop 10 repeated Bid References:")
display(counts.head(10))


Total records: 107,856
Unique Bid References: 78,375
References appearing more than once: 3,574
Records linked to repeated references: 33,055

Top 10 repeated Bid References:


Bid Reference No.
12435513    498
12386800    414
12228273    336
12403271    318
12179407    314
12258712    304
12276629    248
12381345    241
12197037    198
12183520    184
Name: count, dtype: Int64

In [19]:

# Count unique bids in the questionable groups

print("CLOSED WITH AWARD NOTICE")
print("Rows:", len(closed_with_award))
print(
    "Unique Bid References:",
    closed_with_award["Bid Reference No."].nunique()
)

print("\nAWARDED WITH CANCELLED/MISSING NOTICE")
print("Rows:", len(awarded_to_review))
print(
    "Unique Bid References:",
    awarded_to_review["Bid Reference No."].nunique()
)

# Identify available award-related columns
print("\nAvailable award-related columns:")
print([
    c for c in public_df.columns
    if "award" in c.lower()
    or "contract" in c.lower()
    or "lot" in c.lower()
])


CLOSED WITH AWARD NOTICE
Rows: 1581
Unique Bid References: 96

AWARDED WITH CANCELLED/MISSING NOTICE
Rows: 94
Unique Bid References: 46

Available award-related columns:
['Approved Budget of the Contract', 'Contract Duration', 'Award Reference No.', 'Award Title', 'Published Date(Award)', 'Award Date', 'Contract Amount', 'Award Notice Status', 'Contract Effectivity Date', 'Contract End Date', 'Awardee Organization Name', 'Country of Awardee', 'Region of Awardee', 'Province of Awardee', 'City/Municipality of Awardee', 'Awardee Size', 'Awardee Joint Venture']


In [20]:

# Identify bids with more than one Bid Notice Status

bid_status_check = (
    public_df.groupby("Bid Reference No.")[
        "Bid Notice Status"
    ]
    .nunique(dropna=True)
)

conflicting_bids = bid_status_check[
    bid_status_check > 1
]

print(
    "Bid References with multiple statuses:",
    len(conflicting_bids)
)

# Display sample records
sample_ids = conflicting_bids.head(10).index

display(
    public_df[
        public_df["Bid Reference No."].isin(sample_ids)
    ][
        [
            "Bid Reference No.",
            "Bid Notice Status",
            "Award Notice Status",
            "Award Date"
        ]
    ].head(30)
)


Bid References with multiple statuses: 0


,Bid Reference No.,Bid Notice Status,Award Notice Status,Award Date


### 2.5 Audit repeated references and conflicting statuses

The pilot finds exact duplicate rows **and** references repeated because of distinct line items. A repeated reference does not necessarily indicate a duplicate procurement. The analysis checks whether a reference is assigned both Awarded and Failed; in this 2025 Q3 pilot, none is found.

In [21]:

import pandas as pd

id_col = "Bid Reference No."

audit_df = public_df.copy()
audit_df[id_col] = (
    audit_df[id_col].astype("string").str.strip()
)

# Check complete duplicate rows
print("Exact duplicate rows:",
      audit_df.duplicated().sum())

# Create a bid reference-level audit
bid_audit = audit_df.groupby(
    id_col, dropna=False
).agg(
    Row_Count=(id_col, "size"),
    Bid_Status_Count=("Bid Notice Status", "nunique"),
    Award_Status_Count=("Award Notice Status", "nunique")
)

# Check distinct awardees, if available
awardee_col = "Awardee Organisation Name"

if awardee_col in audit_df.columns:
    bid_audit["Distinct_Awardees"] = (
        audit_df.groupby(id_col, dropna=False)[
            awardee_col
        ].nunique()
    )

print("\nReferences with multiple bid statuses:",
      (bid_audit["Bid_Status_Count"] > 1).sum())

print("\nMost frequently repeated bid references:")
display(
    bid_audit.sort_values(
        "Row_Count", ascending=False
    ).head(10)
)

# Inspect the most repeated reference
example_ref = bid_audit["Row_Count"].idxmax()

review_cols = [
    "Bid Reference No.",
    "Bid Notice Status",
    "Award Notice Status",
    "Award Title",
    "Awardee Organisation Name",
    "Award Date",
    "Contract Amount"
]

review_cols = [
    c for c in review_cols
    if c in audit_df.columns
]

print("\nExample bid reference:", example_ref)

display(
    audit_df.loc[
        audit_df[id_col].eq(example_ref),
        review_cols
    ].head(20)
)


Exact duplicate rows: 4794

References with multiple bid statuses: 0

Most frequently repeated bid references:


,Row_Count,Bid_Status_Count,Award_Status_Count
Bid Reference No.,,,
12435513,498,1,0
12386800,414,1,0
12228273,336,1,1
12403271,318,1,0
12179407,314,1,1
12258712,304,1,1
12276629,248,1,0
12381345,241,1,0
12197037,198,1,0



Example bid reference: 12435513


,Bid Reference No.,Bid Notice Status,Award Notice Status,Award Title,Award Date,Contract Amount
105576,12435513,Active,NaN,NaN,NaN,NaN
105577,12435513,Active,NaN,NaN,NaN,NaN
105578,12435513,Active,NaN,NaN,NaN,NaN
105579,12435513,Active,NaN,NaN,NaN,NaN
105580,12435513,Active,NaN,NaN,NaN,NaN
105581,12435513,Active,NaN,NaN,NaN,NaN
105582,12435513,Active,NaN,NaN,NaN,NaN
105583,12435513,Active,NaN,NaN,NaN,NaN
105584,12435513,Active,NaN,NaN,NaN,NaN
105585,12435513,Active,NaN,NaN,NaN,NaN


In [22]:

# Normalize statuses
status = (
    audit_df["Bid Notice Status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Examine unique references with explicit final statuses
for outcome in ["awarded", "failed"]:
    selected = audit_df.loc[status.eq(outcome)]

    print(
        outcome.upper(),
        "| Rows:", f"{len(selected):,}",
        "| Unique bids:",
        f"{selected[id_col].nunique():,}"
    )

# Identify references appearing in both classes
award_ids = set(
    audit_df.loc[status.eq("awarded"), id_col].dropna()
)

failed_ids = set(
    audit_df.loc[status.eq("failed"), id_col].dropna()
)

conflicting_ids = award_ids.intersection(failed_ids)

print(
    "\nReferences marked BOTH Awarded and Failed:",
    len(conflicting_ids)
)


AWARDED | Rows: 16,365 | Unique bids: 10,332
FAILED | Rows: 2,577 | Unique bids: 2,216

References marked BOTH Awarded and Failed: 0


### 2.6 Inspect a highly repeated bid

A heavily repeated reference is examined at the item level to show why hundreds of rows may belong to one procurement. This supports choosing **one row per bid reference** as the final modeling unit rather than treating each item as an independent outcome.

In [23]:

# Investigate the most repeated bid reference

bid_ref = "12435513"

sample = public_df[
    public_df["Bid Reference No."]
    .astype("string")
    .str.strip()
    .eq(bid_ref)
].copy()

columns_to_check = [
    "Bid Reference No.",
    "Notice Title",
    "Line Item No.",
    "Item Name",
    "Item Description",
    "Quantity",
    "Item Budget",
    "Award Reference No.",
    "Award Title",
    "Contract Amount",
    "Bid Notice Status"
]

columns_to_check = [
    c for c in columns_to_check
    if c in sample.columns
]

print("Total rows:", len(sample))

print("\nDistinct values per column:")

display(
    sample[columns_to_check]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)

print("\nSample distinct records:")

display(
    sample[columns_to_check]
    .drop_duplicates()
    .head(15)
)


Total rows: 498

Distinct values per column:


Item Name              498
Item Description       498
Item Budget            335
Quantity                53
Bid Reference No.        1
Notice Title             1
Bid Notice Status        1
Award Reference No.      0
Award Title              0
Contract Amount          0
dtype: int64


Sample distinct records:


,Bid Reference No.,Notice Title,Item Name,Item Description,Quantity,Item Budget,Award Reference No.,Award Title,Contract Amount,Bid Notice Status
105576,12435513,Procurement of Goods Laboratory Supplies for F...,Clean cell (6 x 380 ml per box) Compatible wit...,Clean cell (6 x 380 ml per box) Compatible wit...,45.0,75297.60,NaN,NaN,NaN,Active
105577,12435513,Procurement of Goods Laboratory Supplies for F...,Assay Pro cell (6 x 380 ml per box) Compatible...,Assay Pro cell (6 x 380 ml per box) Compatible...,45.0,80239.05,NaN,NaN,NaN,Active
105578,12435513,Procurement of Goods Laboratory Supplies for F...,Syswash 500 ml/ bottle Compatible with e411 im...,Syswash 500 ml/ bottle Compatible with e411 im...,25.0,32242.75,NaN,NaN,NaN,Active
105579,12435513,Procurement of Goods Laboratory Supplies for F...,Hepatitis B Surface Antigen (HBsAg)- one step ...,Hepatitis B Surface Antigen (HBsAg)- one step ...,8.0,12400.00,NaN,NaN,NaN,Active
105580,12435513,Procurement of Goods Laboratory Supplies for F...,"RPR slide agglutination test with Dilution, 50...","RPR slide agglutination test with Dilution, 50...",3.0,12000.00,NaN,NaN,NaN,Active
105581,12435513,Procurement of Goods Laboratory Supplies for F...,PROCALCITONIN (CalSet & Precicontrols- in kit ...,PROCALCITONIN (CalSet & Precicontrols- in kit ...,30.0,1521000.00,NaN,NaN,NaN,Active
105582,12435513,Procurement of Goods Laboratory Supplies for F...,PerFix-nc (no centrifuge assay kit),PerFix-nc (no centrifuge assay kit),1.0,16640.00,NaN,NaN,NaN,Active
105583,12435513,Procurement of Goods Laboratory Supplies for F...,"Anti streptolysin O reagent (LATEX), 100 test/kit","Anti streptolysin O reagent (LATEX), 100 test/kit",25.0,30000.00,NaN,NaN,NaN,Active
105584,12435513,Procurement of Goods Laboratory Supplies for F...,"C-Reactive protein reagent (CRP) (latex), 100t...","C-Reactive protein reagent (CRP) (latex), 100t...",200.0,494000.00,NaN,NaN,NaN,Active
105585,12435513,Procurement of Goods Laboratory Supplies for F...,"Leptospira IgG / IgM, Immunochromatography tes...","Leptospira IgG / IgM, Immunochromatography tes...",20.0,48000.00,NaN,NaN,NaN,Active


### 2.7 Create a provisional bid-level label table

Statuses are standardized, missing references removed, and a conflict assertion applied. The pilot yields **12,548 uniquely labeled 2025 Q3 bids** (10,332 Awarded; 2,216 Failed). This demonstrates the labeling approach but is **not** the final cross-quarter modeling dataset.

In [24]:

# Create preliminary bid-level outcome table

bid_df = public_df.copy()

bid_df["Bid Reference No."] = (
    bid_df["Bid Reference No."]
    .astype("string")
    .str.strip()
)

bid_df["Bid Notice Status"] = (
    bid_df["Bid Notice Status"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

labeled_bids = bid_df[
    bid_df["Bid Notice Status"].isin(
        ["awarded", "failed"]
    )
].dropna(subset=["Bid Reference No."])

# Confirm no conflicting labels
conflicts = (
    labeled_bids.groupby("Bid Reference No.")[
        "Bid Notice Status"
    ].nunique()
)

assert not (conflicts > 1).any(), \
    "Conflicting labels found. Review before proceeding."

# One provisional outcome per bid
bid_outcomes = (
    labeled_bids[
        ["Bid Reference No.", "Bid Notice Status"]
    ]
    .drop_duplicates()
    .copy()
)

bid_outcomes["Target_Success"] = (
    bid_outcomes["Bid Notice Status"]
    .map({"awarded": 1, "failed": 0})
)

print("Unique labeled bids:", len(bid_outcomes))

display(
    bid_outcomes["Target_Success"]
    .value_counts()
)


Unique labeled bids: 12548


Target_Success
1    10332
0     2216
Name: count, dtype: int64

### 2.8 Pilot bid-level aggregation and provisional EDA

After exact-row deduplication, numeric budget parsing and grouping, the pilot reports one record per bid. It examines the number of item rows, unique items, and whether budgets disagree within the same reference. The next cell filters this provisional table to Awarded/Failed outcomes.

**Scope reminder:** All cells in Phase 2 are a single-quarter methodological pilot. Phase 3 onward expands to every source quarter.

In [25]:

import pandas as pd

# Start from the Public Bidding dataset
data = public_df.copy()

# Standardize bid reference
data["Bid Reference No."] = (
    data["Bid Reference No."]
    .astype("string")
    .str.strip()
)

# Remove rows without a valid reference
data = data.dropna(subset=["Bid Reference No."])
data = data[data["Bid Reference No."] != ""]

# Remove completely identical rows
data = data.drop_duplicates()

# Convert approved contract budget to numeric
budget_col = "Approved Budget of the Contract"

data["ABC_Numeric"] = pd.to_numeric(
    data[budget_col]
    .astype("string")
    .str.replace(r"[^\d.-]", "", regex=True),
    errors="coerce"
)

# Consolidate to one record per bid reference
bid_level = (
    data.groupby("Bid Reference No.", as_index=False)
    .agg(
        Bid_Status=("Bid Notice Status", "first"),
        Notice_Title=("Notice Title", "first"),
        Item_Rows=("Item Name", "size"),
        Distinct_Items=("Item Name", "nunique"),
        Approved_Budget=("ABC_Numeric", "first"),
        Budget_Versions=("ABC_Numeric", "nunique")
    )
)

# Flag potentially inconsistent approved budgets
print(
    "Bids with multiple budget values:",
    (bid_level["Budget_Versions"] > 1).sum()
)

print("Total unique bids:", len(bid_level))

display(bid_level.head(10))


Bids with multiple budget values: 0
Total unique bids: 78375


,Bid Reference No.,Bid_Status,Notice_Title,Item_Rows,Distinct_Items,Approved_Budget,Budget_Versions
0,10072334,Closed,Conduct of GAD related Seminar,1,0,250572.0,1
1,10873299,Closed,CONSTRUCTION / FABRICATION OF IMPOUNDING CAGE,22,20,100000.0,1
2,11128181,Closed,IMPROVEMENT OF WATER SYSTEM,1,0,653435.0,1
3,11301342,Closed,CONSTRUCTION OF ROAD CONCRETE,1,0,450000.0,1
4,11367542,Closed,PUBLIC ADDRESS SYSTEM,1,0,250000.0,1
5,11434444,Closed,Supply and Delivery of Materials for Repair an...,1,0,149099.0,1
6,11487759,Failed,SUPPLY AND DELIVERY OF ESSENTIAL DRUGS AND MED...,1,1,499775.0,1
7,11531992,Closed,CONSTRUCTION OF 1-UNIT MULTIPURPOSE CENTER WIT...,1,0,3000000.0,1
8,11532019,Closed,CONSTRUCTION OF 1-UNIT MULTI-PURPOSE CENTER WI...,1,0,3000000.0,1
9,11532036,Closed,COMMUNITY SUPPORT PROGRAM FOR CSP BARANGAY THR...,1,0,30000000.0,1


In [26]:

# Examine all bid-level statuses
print("Bid-level status distribution:")

display(
    bid_level["Bid_Status"]
    .value_counts(dropna=False)
)

# Prepare provisional labeled sample
labeled_bids = bid_level[
    bid_level["Bid_Status"]
    .astype("string")
    .str.strip()
    .str.casefold()
    .isin(["awarded", "failed"])
].copy()

labeled_bids["Target_Success"] = (
    labeled_bids["Bid_Status"]
    .str.strip()
    .str.casefold()
    .map({"awarded": 1, "failed": 0})
)

print("\nProvisional labeled bids:", len(labeled_bids))
display(labeled_bids["Target_Success"].value_counts())


Bid-level status distribution:


Bid_Status
Closed         51907
Awarded        10332
Active         10131
Cancelled       3741
Failed          2216
Shortlisted       48
Name: count, dtype: int64


Provisional labeled bids: 12548


Target_Success
1    10332
0     2216
Name: count, dtype: int64

## Phase 3. Extend coverage and diagnose cross-year data quality

**Purpose:** Scale from the Q3 pilot to the full 14-quarter history and identify schema or labeling breaks before combining files. File sizes and quarter completeness are inspected first.

In [27]:

from pathlib import Path
import pandas as pd

files = sorted(
    output_dir.glob("PHILGEPS_20??_Q[1-4].csv")
)

inventory = pd.DataFrame({
    "File": [f.name for f in files],
    "Size_MB": [
        round(f.stat().st_size / (1024**2), 2)
        for f in files
    ]
})

display(inventory)
print("Total quarterly files:", len(files))

assert len(files) == 14, (
    "Expected 14 quarterly source files."
)


,File,Size_MB
0,PHILGEPS_2023_Q1.csv,278.61
1,PHILGEPS_2023_Q2.csv,295.77
2,PHILGEPS_2023_Q3.csv,308.73
3,PHILGEPS_2023_Q4.csv,312.78
4,PHILGEPS_2024_Q1.csv,306.27
5,PHILGEPS_2024_Q2.csv,329.33
6,PHILGEPS_2024_Q3.csv,324.69
7,PHILGEPS_2024_Q4.csv,355.77
8,PHILGEPS_2025_Q1.csv,321.44
9,PHILGEPS_2025_Q2.csv,251.67


Total quarterly files: 13


AssertionError: Expected 14 quarterly source files.

### 3.1 Preliminary quarter-level inventory and outcome audit

The first all-quarter scan counts Public Bidding rows, unique references and notice statuses. Its strict `public bidding` match is later shown to **miss 2026 records** labeled `Competitive Bidding (Public Bidding)`. The audit also predates the repaired 2025 Q4 schema. **Do not use the preliminary totals as the final project population.**

In [ ]:

import pandas as pd
from pathlib import Path
from collections import Counter

files = sorted(
    output_dir.glob("PHILGEPS_20??_Q[1-4].csv")
)

required_cols = [
    "Bid Reference No.",
    "Procurement Mode",
    "Bid Notice Status"
]

results = []
all_bid_ids = set()
cross_quarter_ids = set()

for file in files:
    total_rows = 0
    public_rows = 0
    quarter_ids = set()
    status_counts = Counter()

    # Check required columns
    header = pd.read_csv(
        file, nrows=0, encoding="utf-8-sig"
    ).columns.tolist()

    missing = [
        c for c in required_cols if c not in header
    ]

    if missing:
        raise ValueError(
            f"{file.name}: Missing columns {missing}"
        )

    for chunk in pd.read_csv(
        file,
        usecols=required_cols,
        chunksize=50000,
        encoding="utf-8-sig",
        dtype="string",
        on_bad_lines="error"
    ):
        total_rows += len(chunk)

        mode = (
            chunk["Procurement Mode"]
            .str.strip()
            .str.casefold()
        )

        public = chunk.loc[
            mode.eq("public bidding").fillna(False)
        ]

        public_rows += len(public)

        ids = (
            public["Bid Reference No."]
            .dropna()
            .str.strip()
        )

        quarter_ids.update(
            ids[ids.ne("")].tolist()
        )

        statuses = (
            public["Bid Notice Status"]
            .str.strip()
            .str.casefold()
            .value_counts()
        )

        status_counts.update(
            statuses.to_dict()
        )

    cross_quarter_ids.update(
        quarter_ids.intersection(all_bid_ids)
    )
    all_bid_ids.update(quarter_ids)

    results.append({
        "Quarter": file.stem,
        "All_Records": total_rows,
        "Public_Bidding_Rows": public_rows,
        "Unique_Bids": len(quarter_ids),
        "Awarded_Rows": status_counts["awarded"],
        "Failed_Rows": status_counts["failed"],
        "Closed_Rows": status_counts["closed"],
        "Active_Rows": status_counts["active"],
        "Cancelled_Rows": status_counts["cancelled"]
    })

    print(f"Completed: {file.name}")

# Display audit results
audit_df = pd.DataFrame(results)

display(audit_df)

print("\nTOTAL PUBLIC BIDDING ROWS:",
      f"{audit_df['Public_Bidding_Rows'].sum():,}")

print("UNIQUE BID REFERENCES ACROSS ALL QUARTERS:",
      f"{len(all_bid_ids):,}")

print("BID REFERENCES APPEARING IN MULTIPLE QUARTERS:",
      f"{len(cross_quarter_ids):,}")

# Save audit results
audit_df.to_csv(
    output_dir.parent / "PHILGEPS_Quarterly_Audit.csv",
    index=False
)


### 3.2 Investigate unexpected changes in 2025 Q4 and 2026

The checks reveal two distinct compatibility problems:

- **2025 Q4:** `Bid Notice Status` appears mostly blank or numeric, suggesting misalignment between source values and headers.
- **2026 Q1/Q2:** The procurement method is named **Competitive Bidding (Public Bidding)**, not exactly **Public Bidding**, so an old exact-match filter produces false zero counts.

The next steps inspect these issues rather than assuming procurement activity disappeared.

In [ ]:

import pandas as pd
from collections import Counter

files_to_check = [
    "PHILGEPS_2025_Q4.csv",
    "PHILGEPS_2026_Q1.csv",
    "PHILGEPS_2026_Q2.csv"
]

for filename in files_to_check:

    path = output_dir / filename

    print("\n" + "=" * 55)
    print("FILE:", filename)

    header = pd.read_csv(
        path, nrows=0, encoding="utf-8-sig"
    ).columns.tolist()

    print("Total columns:", len(header))

    fields = [
        "Procurement Mode",
        "Bid Notice Status"
    ]

    print("Missing required columns:",
          [c for c in fields if c not in header])

    if any(c not in header for c in fields):
        continue

    mode_counts = Counter()
    status_counts = Counter()
    public_status_counts = Counter()
    total = 0
    public_total = 0

    for chunk in pd.read_csv(
        path,
        usecols=fields,
        chunksize=50000,
        dtype="string",
        encoding="utf-8-sig"
    ):

        total += len(chunk)

        mode = (
            chunk["Procurement Mode"]
            .fillna("<MISSING>")
            .str.strip()
        )

        status = (
            chunk["Bid Notice Status"]
            .fillna("<MISSING>")
            .str.strip()
        )

        mode_counts.update(mode.value_counts().to_dict())
        status_counts.update(status.value_counts().to_dict())

        public_mask = mode.str.casefold().eq(
            "public bidding"
        )

        public_total += public_mask.sum()

        public_status_counts.update(
            status.loc[public_mask]
            .value_counts()
            .to_dict()
        )

    print("Total rows:", f"{total:,}")
    print("Exact Public Bidding rows:",
          f"{public_total:,}")

    print("\nTOP PROCUREMENT MODES:")
    display(
        pd.Series(mode_counts)
        .sort_values(ascending=False)
        .head(15)
        .to_frame("Records")
    )

    print("\nBID NOTICE STATUS DISTRIBUTION:")
    display(
        pd.Series(status_counts)
        .sort_values(ascending=False)
        .head(15)
        .to_frame("Records")
    )

    print("\nSTATUS OF EXACT PUBLIC BIDDING ROWS:")
    display(
        pd.Series(public_status_counts)
        .sort_values(ascending=False)
        .head(15)
        .to_frame("Records")
    )


### 3.3 Compare fields and inspect suspect 2025 Q4 rows

Header positions and individual records are compared against 2025 Q3. The Q4 anomaly is not merely a new spelling: the apparent status value sits under another field. These diagnostic cells help locate the shift before any correction is applied.

In [ ]:

import pandas as pd

files_to_compare = [
    "PHILGEPS_2025_Q3.csv",
    "PHILGEPS_2025_Q4.csv",
    "PHILGEPS_2026_Q1.csv"
]

schemas = {}

for filename in files_to_compare:
    path = output_dir / filename

    columns = pd.read_csv(
        path,
        nrows=0,
        encoding="utf-8-sig"
    ).columns.tolist()

    schemas[filename] = columns

    print("\nFILE:", filename)
    print("Number of columns:", len(columns))

    if "Bid Notice Status" in columns:
        idx = columns.index("Bid Notice Status")

        print("\nColumns around Bid Notice Status:")

        for i in range(
            max(0, idx - 4),
            min(len(columns), idx + 5)
        ):
            print(i, columns[i])

q3 = set(schemas["PHILGEPS_2025_Q3.csv"])
q4 = set(schemas["PHILGEPS_2025_Q4.csv"])
q1_2026 = set(schemas["PHILGEPS_2026_Q1.csv"])

print("\nColumns only in 2025 Q4 vs Q3:")
print(sorted(q4 - q3))

print("\nColumns in Q3 but not Q4:")
print(sorted(q3 - q4))

print("\nColumns only in 2026 Q1 vs 2025 Q4:")
print(sorted(q1_2026 - q4))


In [ ]:

q4_path = output_dir / "PHILGEPS_2025_Q4.csv"

missing_sample = pd.DataFrame()
numeric_sample = pd.DataFrame()

for chunk in pd.read_csv(
    q4_path,
    chunksize=50000,
    encoding="utf-8-sig",
    dtype="string"
):
    mode = (
        chunk["Procurement Mode"]
        .str.strip()
        .str.casefold()
    )

    public = chunk.loc[
        mode.eq("public bidding").fillna(False)
    ]

    status = public["Bid Notice Status"]

    if missing_sample.empty:
        missing_sample = public.loc[
            status.isna() | status.str.strip().eq("").fillna(False)
        ].head(5).copy()

    if numeric_sample.empty:
        numeric_sample = public.loc[
            status.str.fullmatch(r"\d+").fillna(False)
        ].head(5).copy()

    if not missing_sample.empty and not numeric_sample.empty:
        break

review_cols = [
    "Bid Reference No.",
    "Procurement Mode",
    "Bid Notice Status",
    "Award Notice Status",
    "Award Reference No.",
    "Award Date",
    "Notice Title"
]

review_cols = [
    c for c in review_cols
    if c in schemas["PHILGEPS_2025_Q4.csv"]
]

print("Q4: Records with missing status")
display(missing_sample[review_cols])

print("\nQ4: Records with numeric status")
display(numeric_sample[review_cols])


### 3.4 Expand the Public Bidding filter definition

The accepted mode set includes both `public bidding` and `competitive bidding (public bidding)`. This short cell is a **historical exploratory snippet** using a previously created `chunk`, not a complete independent processing step. The finalized consolidation later applies the rule consistently for every file.

In [ ]:

PUBLIC_BIDDING_MODES = {
    "public bidding",
    "competitive bidding (public bidding)"
}

# Use inside each processing chunk
mode = (
    chunk["Procurement Mode"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

public_mask = mode.isin(PUBLIC_BIDDING_MODES)

public = chunk.loc[public_mask].copy()


In [ ]:

import pandas as pd

q4_path = output_dir / "PHILGEPS_2025_Q4.csv"

missing_examples = []
numeric_examples = []

for chunk in pd.read_csv(
    q4_path,
    chunksize=50000,
    dtype="string",
    encoding="utf-8-sig"
):
    mode = (
        chunk["Procurement Mode"]
        .str.strip()
        .str.casefold()
    )

    public = chunk.loc[
        mode.eq("public bidding").fillna(False)
    ]

    status = public["Bid Notice Status"]

    # Collect examples with missing status
    if len(missing_examples) < 3:
        examples = public.loc[
            status.isna() |
            status.str.strip().eq("").fillna(False)
        ].head(3 - len(missing_examples))

        missing_examples.extend(
            examples.to_dict("records")
        )

    # Collect examples with numeric status
    if len(numeric_examples) < 3:
        examples = public.loc[
            status.str.fullmatch(r"\d+").fillna(False)
        ].head(3 - len(numeric_examples))

        numeric_examples.extend(
            examples.to_dict("records")
        )

    if len(missing_examples) == 3 and len(numeric_examples) == 3:
        break

# Display neighboring fields vertically
columns = pd.read_csv(
    q4_path, nrows=0
).columns.tolist()

review_cols = [
    "Bid Reference No."
] + columns[24:39]

print("Q4 RECORDS WITH MISSING STATUS")
display(
    pd.DataFrame(missing_examples)
    [review_cols].T
)

print("\nQ4 RECORDS WITH NUMERIC STATUS")
display(
    pd.DataFrame(numeric_examples)
    [review_cols].T
)


### 3.5 Investigate Q4 status values and missing fields

Several exploratory diagnostics test whether status strings have moved into the wrong position, and whether apparently missing values reflect a malformed source structure. These steps guide the repair; they should not be interpreted as evidence that most procurements have missing outcomes.

In [ ]:

import pandas as pd
from collections import Counter

q4_path = output_dir / "PHILGEPS_2025_Q4.csv"

valid_statuses = {
    "awarded", "failed", "closed",
    "active", "cancelled", "shortlisted",
    "published", "concluded"
}

counts = Counter()
budget_status_counts = Counter()
samples = {}

for chunk in pd.read_csv(
    q4_path,
    chunksize=50000,
    dtype="string",
    encoding="utf-8-sig"
):
    mode = chunk["Procurement Mode"].str.strip().str.casefold()

    public = chunk.loc[
        mode.eq("public bidding").fillna(False)
    ].copy()

    status = public["Bid Notice Status"].str.strip()
    budget = public["Item Budget"].str.strip().str.casefold()

    # Detect possible misplaced status
    status_in_budget = budget.isin(valid_statuses)
    numeric_status = status.str.fullmatch(
        r"\d+"
    ).fillna(False)
    missing_status = status.isna() | status.eq("").fillna(False)

    counts["Public Bidding Rows"] += len(public)

    counts["Status in Item Budget"] += (
        status_in_budget.sum()
    )

    counts["Status in Budget + Numeric Status"] += (
        (status_in_budget & numeric_status).sum()
    )

    counts["Status in Budget + Missing Status"] += (
        (status_in_budget & missing_status).sum()
    )

    counts["Missing Status Without Budget Label"] += (
        (missing_status & ~status_in_budget).sum()
    )

    budget_status_counts.update(
        budget.loc[status_in_budget]
        .value_counts().to_dict()
    )

    groups = {
        "Missing status with budget label":
            missing_status & status_in_budget,
        "Missing status without budget label":
            missing_status & ~status_in_budget
    }

    for name, mask in groups.items():
        if name not in samples:
            example = public.loc[mask].head(3)
            if not example.empty:
                samples[name] = example

print("Q4 DATA ALIGNMENT AUDIT")
display(pd.Series(counts).to_frame("Records"))

print("\nStatuses found under Item Budget:")
display(
    pd.Series(budget_status_counts)
    .sort_values(ascending=False)
    .to_frame("Records")
)

review_cols = [
    "Bid Reference No.",
    "Item Description",
    "Quantity",
    "UOM",
    "Item Budget",
    "Bid Notice Status",
    "Award Reference No.",
    "Award Title",
    "UNSPSC Code"
]

for name, sample in samples.items():
    print("\n", name)
    display(sample[review_cols].T)


In [ ]:

import pandas as pd

q4_path = output_dir / "PHILGEPS_2025_Q4.csv"

# Read only the columns needed for diagnosis
q4_check = pd.read_csv(
    q4_path,
    usecols=[
        "Procurement Mode",
        "Item Budget",
        "Bid Notice Status"
    ],
    dtype="string",
    encoding="utf-8-sig"
)

# Filter Public Bidding
mode = (
    q4_check["Procurement Mode"]
    .str.strip()
    .str.casefold()
)

q4_public = q4_check.loc[
    mode.eq("public bidding").fillna(False)
].copy()

# Valid status categories
valid_statuses = {
    "awarded",
    "failed",
    "closed",
    "active",
    "cancelled",
    "shortlisted",
    "published",
    "concluded"
}

# Inspect both columns
q4_public["Budget_Status"] = (
    q4_public["Item Budget"]
    .str.strip()
    .str.casefold()
)

q4_public["Original_Status"] = (
    q4_public["Bid Notice Status"]
    .str.strip()
    .str.casefold()
)

print("Total Public Bidding rows:", len(q4_public))

print("\nSTATUS VALUES FOUND UNDER ITEM BUDGET:")
display(
    q4_public.loc[
        q4_public["Budget_Status"].isin(valid_statuses),
        "Budget_Status"
    ].value_counts()
)

recoverable = (
    q4_public["Budget_Status"].isin(valid_statuses)
    & ~q4_public["Original_Status"].isin(valid_statuses)
)

print("\nPotentially recoverable statuses:",
      f"{recoverable.sum():,}")

print("Still unresolved:",
      f"{len(q4_public) - recoverable.sum() - (q4_public['Original_Status'].isin(valid_statuses) & ~recoverable).sum():,}")


### 3.6 Preliminary repair hypotheses and raw-record verification

The exploratory cleanup is followed by inspection of the original text's delimiter-separated records. Although the rows have **46 fields**, the 2025 Q4 data omit the *Calendar Type* value and shift subsequent values left by one position. A matching field count alone therefore failed to detect the error. The final guarded repair is applied in the next subsection.

In [ ]:

import pandas as pd
from pathlib import Path

source = output_dir / "PHILGEPS_2025_Q4.csv"

clean_dir = output_dir.parent / "dataset_cleaned"
clean_dir.mkdir(exist_ok=True)

# Read only the fields needed for outcome recovery
q4 = pd.read_csv(
    source,
    usecols=[
        "Bid Reference No.",
        "Procurement Mode",
        "Item Budget",
        "Bid Notice Status"
    ],
    dtype="string",
    encoding="utf-8-sig"
)

# Public Bidding only
mode = (
    q4["Procurement Mode"]
    .str.strip()
    .str.casefold()
)

q4 = q4.loc[
    mode.eq("public bidding").fillna(False)
].copy()

# Preserve the original fields
q4["Original_Bid_Status"] = q4["Bid Notice Status"]

# Recover status from the misplaced column
q4["Recovered_Bid_Status"] = (
    q4["Item Budget"]
    .str.strip()
    .str.casefold()
)

valid_statuses = {
    "awarded", "failed", "closed",
    "active", "cancelled", "shortlisted"
}

q4["Status_Valid"] = (
    q4["Recovered_Bid_Status"].isin(valid_statuses)
)

q4["Bid Reference No."] = (
    q4["Bid Reference No."]
    .str.strip()
)

print("Total Public Bidding rows:", len(q4))
print("Valid recovered statuses:",
      q4["Status_Valid"].sum())

# Check for missing IDs
print("Missing bid references:",
      q4["Bid Reference No."].isna().sum())

# Check inconsistent statuses by bid
status_check = (
    q4.dropna(subset=["Bid Reference No."])
    .groupby("Bid Reference No.")[
        "Recovered_Bid_Status"
    ]
    .nunique()
)

conflicts = (status_check > 1).sum()

print("References with conflicting statuses:", conflicts)

# Summarize all recovered outcomes
print("\nRecovered status distribution:")
display(q4["Recovered_Bid_Status"].value_counts())

# Save audit without modifying original CSV
audit_path = clean_dir / "Q4_2025_Status_Recovery_Audit.csv"

q4.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig"
)

# Construct provisional bid-level labels
if q4["Status_Valid"].all() and conflicts == 0:

    labeled = q4.loc[
        q4["Recovered_Bid_Status"].isin(
            ["awarded", "failed"]
        )
    ].dropna(subset=["Bid Reference No."]).copy()

    labeled = labeled[
        labeled["Bid Reference No."].ne("")
    ]

    labeled = labeled[
        ["Bid Reference No.", "Recovered_Bid_Status"]
    ].drop_duplicates()

    labeled["Target_Success"] = (
        labeled["Recovered_Bid_Status"]
        .map({"awarded": 1, "failed": 0})
    )

    print("\nUnique labeled bidding references:",
          len(labeled))

    display(
        labeled["Target_Success"].value_counts()
    )

    labeled.to_csv(
        clean_dir / "Q4_2025_Provisional_Labels.csv",
        index=False,
        encoding="utf-8-sig"
    )

else:
    print(
        "\nReview invalid or conflicting statuses"
        " before creating bid-level labels."
    )

print("\nRecovery audit saved to:", audit_path)


In [ ]:

import csv
from pathlib import Path
from collections import Counter

source_folder = output_dir.parent / "dataset"

# Locate original Q4 2025 TXT
matches = [
    p for p in source_folder.iterdir()
    if p.stem.casefold() == "2025_oct-dec"
    and p.suffix.lower() in [".txt", ".csv"]
]

if not matches:
    raise FileNotFoundError(
        "Original 2025_Oct-Dec file not found."
    )

source_txt = matches[0]

csv.field_size_limit(10_000_000)

# Detect delimiter
with open(
    source_txt,
    "r",
    encoding="utf-8-sig",
    errors="replace",
    newline=""
) as f:
    sample = f.read(65536)

dialect = csv.Sniffer().sniff(
    sample,
    delimiters=",\t;|"
)

# Audit raw field counts
field_counts = Counter()
examples = {}

with open(
    source_txt,
    "r",
    encoding="utf-8-sig",
    errors="replace",
    newline=""
) as f:

    reader = csv.reader(
        f, delimiter=dialect.delimiter
    )

    header = next(reader)
    expected = len(header)

    for row_number, row in enumerate(reader, start=2):
        field_counts[len(row)] += 1

        if len(row) not in examples:
            examples[len(row)] = (row_number, row)

print("Expected columns:", expected)

print("\nRaw record field counts:")
display(
    pd.Series(field_counts)
    .sort_index()
    .to_frame("Records")
)

print("\nExamples by field count:")

for count, (row_number, row) in examples.items():

    print(f"\nRecord {row_number}: {count} fields")

    for i in range(22, min(len(row), 31)):
        print(
            i,
            header[i] if i < len(header) else "EXTRA",
            ":",
            str(row[i])[:80]
        )


In [ ]:

import pandas as pd

def get_awarded_sample(filename, status_column):

    path = output_dir / filename

    for chunk in pd.read_csv(
        path,
        chunksize=50000,
        dtype="string",
        encoding="utf-8-sig"
    ):
        mode = (
            chunk["Procurement Mode"]
            .str.strip()
            .str.casefold()
        )

        status = (
            chunk[status_column]
            .str.strip()
            .str.casefold()
        )

        mask = (
            mode.eq("public bidding").fillna(False)
            & status.eq("awarded").fillna(False)
        )

        if mask.any():
            return chunk.loc[mask].iloc[0]

    return None


# Reference: correctly interpreted Q3 dataset
q3_sample = get_awarded_sample(
    "PHILGEPS_2025_Q3.csv",
    "Bid Notice Status"
)

# Q4: status appears under Item Budget
q4_sample = get_awarded_sample(
    "PHILGEPS_2025_Q4.csv",
    "Item Budget"
)

if q3_sample is None or q4_sample is None:
    raise ValueError("Could not locate both sample records.")

comparison = pd.DataFrame({
    "Position": range(len(q3_sample.index)),
    "Column": q3_sample.index,
    "Q3_2025_Value": q3_sample.values,
    "Q4_2025_Value": q4_sample.values
})

display(comparison.iloc[17:46])


### 3.7 Validated 2025 Q4 schema repair

**Corrective method:** Before writing a corrected file, the code checks that shifted status values belong to recognized labels and that the final source field has no meaningful trailing information. It then inserts an empty `Calendar Type` position and moves the affected fields one column to the right, creating `PHILGEPS_2025_Q4_CLEAN.csv`.

The saved audit reports **352,091** source rows, **zero** unrecognized shifted statuses and **zero** nonempty trailing fields. This is a documented data-repair decision, not an arbitrary replacement of labels.

In [ ]:

import pandas as pd
from pathlib import Path
from collections import Counter

source = output_dir / "PHILGEPS_2025_Q4.csv"
clean_dir = output_dir.parent / "dataset_cleaned"
clean_dir.mkdir(exist_ok=True)

destination = clean_dir / "PHILGEPS_2025_Q4_CLEAN.csv"

headers = pd.read_csv(
    source, nrows=0, encoding="utf-8-sig"
).columns.tolist()

assert len(headers) == 46
assert headers[20] == "Calendar Type"
assert headers[26] == "Item Budget"
assert headers[27] == "Bid Notice Status"

valid_statuses = {
    "awarded", "failed", "closed",
    "active", "cancelled", "shortlisted",
    "published", "concluded", "cancel_approved"
}

status_counts = Counter()
invalid_status = 0
trailing_nonempty = 0
total_rows = 0

# PASS 1: Validate all records before repair
for chunk in pd.read_csv(
    source,
    chunksize=30000,
    dtype="string",
    keep_default_na=False,
    encoding="utf-8-sig"
):

    total_rows += len(chunk)

    shifted_status = (
        chunk["Item Budget"]
        .str.strip()
        .str.casefold()
    )

    status_counts.update(
        shifted_status.value_counts().to_dict()
    )

    invalid_status += (
        ~shifted_status.isin(valid_statuses)
    ).sum()

    # Check whether the final source field
    # contains meaningful information
    last_field = (
        chunk.iloc[:, 45]
        .str.strip()
        .str.casefold()
    )

    empty_values = {
        "", "null", "none", "nan", "<na>"
    }

    trailing_nonempty += (
        ~last_field.isin(empty_values)
    ).sum()

print("Total records:", f"{total_rows:,}")
print("Unrecognized shifted statuses:", invalid_status)
print("Nonempty trailing values:", trailing_nonempty)

print("\nStatus distribution:")
display(
    pd.Series(status_counts)
    .sort_values(ascending=False)
    .to_frame("Records")
)

# PASS 2: Repair only if validation passes
if invalid_status == 0 and trailing_nonempty == 0:

    temp_path = destination.with_suffix(".tmp.csv")
    first_chunk = True

    for chunk in pd.read_csv(
        source,
        chunksize=30000,
        dtype="string",
        keep_default_na=False,
        encoding="utf-8-sig"
    ):

        # Preserve columns 0 through 19
        left = chunk.iloc[:, :20].copy()

        # Insert missing Calendar Type
        left["Calendar Type"] = pd.NA

        # Shift original columns 20-44 one
        # position to the right
        right = chunk.iloc[:, 20:45].copy()
        right.columns = headers[21:46]

        repaired = pd.concat(
            [left, right], axis=1
        )

        assert repaired.columns.tolist() == headers

        repaired.to_csv(
            temp_path,
            mode="w" if first_chunk else "a",
            header=first_chunk,
            index=False,
            encoding="utf-8-sig"
        )

        first_chunk = False

    temp_path.replace(destination)

    print("\nQ4 repair completed.")
    print("Saved to:", destination)

else:
    print("\nREPAIR STOPPED.")
    print("Review inconsistent records before shifting.")


### 3.8 Validate the corrected Q4 file

After repair, the Public Bidding subset contains **115,632** item rows with recognizable statuses, including **14,397 Awarded** and **2,927 Failed** rows. The additional check verifies reference-level consistency and identifies item-budget fields that remain missing or nonnumeric; those item fields are not blindly used as model inputs.

In [ ]:

clean_q4 = pd.read_csv(
    destination,
    usecols=[
        "Procurement Mode",
        "Bid Notice Status",
        "Item Budget",
        "Quantity",
        "UOM",
        "Calendar Type"
    ],
    dtype="string",
    encoding="utf-8-sig"
)

public_mask = (
    clean_q4["Procurement Mode"]
    .str.strip()
    .str.casefold()
    .eq("public bidding")
)

q4_public_clean = clean_q4.loc[public_mask]

print("Public Bidding rows:", len(q4_public_clean))

print("\nCorrected status distribution:")
display(
    q4_public_clean["Bid Notice Status"]
    .value_counts(dropna=False)
)

print("\nSample corrected data:")
display(q4_public_clean.head(10))


In [ ]:

import pandas as pd
from pathlib import Path
from collections import Counter

clean_file = (
    output_dir.parent
    / "dataset_cleaned"
    / "PHILGEPS_2025_Q4_CLEAN.csv"
)

status_counts = Counter()
bid_statuses = {}
total_public = 0
invalid_budgets = 0

for chunk in pd.read_csv(
    clean_file,
    usecols=[
        "Bid Reference No.",
        "Procurement Mode",
        "Bid Notice Status",
        "Item Budget"
    ],
    chunksize=50000,
    dtype="string",
    encoding="utf-8-sig"
):

    mode = (
        chunk["Procurement Mode"]
        .str.strip()
        .str.casefold()
    )

    public = chunk.loc[
        mode.eq("public bidding").fillna(False)
    ].copy()

    total_public += len(public)

    status = (
        public["Bid Notice Status"]
        .str.strip()
        .str.casefold()
    )

    status_counts.update(status.value_counts().to_dict())

    budget = pd.to_numeric(
        public["Item Budget"]
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

    invalid_budgets += budget.isna().sum()

    for ref, st in zip(
        public["Bid Reference No."],
        status
    ):
        if pd.notna(ref) and pd.notna(st):
            ref = str(ref).strip()
            if ref:
                bid_statuses.setdefault(ref, set()).add(st)

conflicts = sum(
    len(status_set) > 1
    for status_set in bid_statuses.values()
)

print("PUBLIC BIDDING VALIDATION")
print("Total rows:", f"{total_public:,}")
print("Unique bid references:", f"{len(bid_statuses):,}")
print("Conflicting bid statuses:", conflicts)
print("Non-numeric/missing Item Budgets:", invalid_budgets)

print("\nSTATUS DISTRIBUTION")
display(
    pd.Series(status_counts)
    .sort_values(ascending=False)
    .to_frame("Records")
)

print("\nExpected rows: 115,632")
print("Matches:", total_public == 115632)


## Phase 4. Consolidate and standardize the 14 quarters

**Purpose:** Produce a single item-level source table with a consistent schema while preserving lineage. The process:

1. Replaces only the affected 2025 Q4 source with the repaired file.
2. Recognizes both 2023–2025 and 2026 names for Public Bidding.
3. Maps alternative source headers to standard fields, e.g., bid reference, status, approved budget, funding source, category, region and publication date.
4. Adds `Source_Quarter` and a `Data_Quality_Flag` identifying repaired Q4 data.
5. Writes a consolidated CSV in manageable chunks.

**Recorded consolidated total:** **1,840,546 item-level Public Bidding rows.** These are *not* 1,840,546 unique procurements.

In [ ]:

import pandas as pd
from pathlib import Path

clean_dir = output_dir.parent / "dataset_cleaned"
clean_dir.mkdir(exist_ok=True)

quarter_files = sorted(
    output_dir.glob("PHILGEPS_20??_Q[1-4].csv")
)

assert len(quarter_files) == 14

# Standardized column mapping
column_map = {
    "Bid_Reference": ["Bid Reference No."],
    "Bid_Status": ["Bid Notice Status"],
    "Notice_Title": ["Notice Title"],
    "Funding_Source": [
        "Funding Source", "Source of Funds"
    ],
    "Approved_Budget": [
        "Approved Budget of the Contract",
        "Approved Budget for the Contract (ABC)"
    ],
    "Item_Name": [
        "Item Name", "Item/Lot Name"
    ],
    "Item_Budget": ["Item Budget"],
    "Procurement_Category": [
        "Classification",
        "Procurement Category"
    ],
    "Region": ["Region"],
    "Published_Date": ["Published Date"]
}

public_modes = {
    "public bidding",
    "competitive bidding (public bidding)"
}

output_file = (
    clean_dir / "PHILGEPS_PUBLIC_BIDDING_ALL.csv"
)

summary = []
first_write = True

for original_file in quarter_files:

    # Use repaired 2025 Q4 file
    if original_file.stem == "PHILGEPS_2025_Q4":
        source = (
            clean_dir / "PHILGEPS_2025_Q4_CLEAN.csv"
        )
    else:
        source = original_file

    if not source.exists():
        raise FileNotFoundError(source)

    headers = pd.read_csv(
        source, nrows=0, encoding="utf-8-sig"
    ).columns.tolist()

    header_lookup = {
        h.strip().casefold(): h for h in headers
    }

    resolved = {}

    for standard_name, alternatives in column_map.items():
        resolved[standard_name] = next(
            (
                header_lookup[name.casefold()]
                for name in alternatives
                if name.casefold() in header_lookup
            ),
            None
        )

    required = ["Bid_Reference", "Bid_Status"]

    for field in required:
        if resolved[field] is None:
            raise ValueError(
                f"{source.name}: Missing {field}"
            )

    if "Procurement Mode" not in headers:
        raise ValueError(
            f"{source.name}: Procurement Mode missing"
        )

    usecols = list(set(
        [c for c in resolved.values() if c is not None]
        + ["Procurement Mode"]
    ))

    public_rows = 0

    for chunk in pd.read_csv(
        source,
        usecols=usecols,
        chunksize=50000,
        dtype="string",
        encoding="utf-8-sig"
    ):

        mode = (
            chunk["Procurement Mode"]
            .str.strip()
            .str.casefold()
        )

        public = chunk.loc[
            mode.isin(public_modes)
        ].copy()

        if public.empty:
            continue

        standardized = pd.DataFrame(
            index=public.index
        )

        for field, source_col in resolved.items():
            if source_col is not None:
                standardized[field] = public[source_col]
            else:
                standardized[field] = pd.NA

        standardized["Source_Quarter"] = (
            original_file.stem.replace("PHILGEPS_", "")
        )

        standardized["Data_Quality_Flag"] = (
            "Q4_2025_SCHEMA_REPAIRED"
            if original_file.stem == "PHILGEPS_2025_Q4"
            else "ORIGINAL"
        )

        standardized["Bid_Reference"] = (
            standardized["Bid_Reference"].str.strip()
        )

        standardized["Bid_Status"] = (
            standardized["Bid_Status"]
            .str.strip()
            .str.casefold()
        )

        standardized.to_csv(
            output_file,
            mode="w" if first_write else "a",
            header=first_write,
            index=False,
            encoding="utf-8-sig"
        )

        first_write = False
        public_rows += len(public)

    summary.append({
        "Quarter": original_file.stem,
        "Public_Bidding_Rows": public_rows
    })

    print(
        f"Completed: {original_file.stem} "
        f"| {public_rows:,} rows"
    )

if first_write:
    raise ValueError("No Public Bidding records found.")

summary_df = pd.DataFrame(summary)

print("\nCONSOLIDATION SUMMARY")
display(summary_df)

print(
    "\nTotal Public Bidding rows:",
    f"{summary_df['Public_Bidding_Rows'].sum():,}"
)

print("\nSaved to:", output_file)


## Phase 5. Create and audit one observation per procurement

### 5.1 Use SQLite to process large volumes

The consolidated item-level file is too large to treat as a small notebook table. A SQLite audit database stores bid references, statuses and source quarters; this enables checks for unique references, cross-quarter repeats and conflicts.

**Recorded audit:** **1,055,442 unique bid references** across the consolidated data, with **two** cross-quarter references. The next step derives exactly one outcome per reference.

In [ ]:

import pandas as pd
import sqlite3
from pathlib import Path

source = (
    output_dir.parent
    / "dataset_cleaned"
    / "PHILGEPS_PUBLIC_BIDDING_ALL.csv"
)

db_path = (
    output_dir.parent
    / "dataset_cleaned"
    / "PHILGEPS_BID_AUDIT.sqlite"
)

con = sqlite3.connect(db_path)

# Rebuild the temporary audit table
con.execute("DROP TABLE IF EXISTS bid_audit")

total = 0

for chunk in pd.read_csv(
    source,
    usecols=[
        "Bid_Reference",
        "Bid_Status",
        "Source_Quarter"
    ],
    dtype="string",
    chunksize=50000,
    encoding="utf-8-sig"
):

    for col in ["Bid_Reference", "Bid_Status"]:
        chunk[col] = chunk[col].str.strip()

    chunk["Bid_Status"] = (
        chunk["Bid_Status"].str.casefold()
    )

    chunk["Bid_Reference"] = (
        chunk["Bid_Reference"]
        .replace(["", "NULL", "None", "nan"], pd.NA)
    )

    chunk["Bid_Status"] = (
        chunk["Bid_Status"]
        .replace(["", "null", "none", "nan"], pd.NA)
    )

    chunk.to_sql(
        "bid_audit",
        con,
        if_exists="append",
        index=False
    )

    total += len(chunk)

print("Records processed:", f"{total:,}")

# Improve query performance
con.execute(
    "CREATE INDEX idx_bid_ref "
    "ON bid_audit(Bid_Reference)"
)

# Calculate unique references
summary = pd.read_sql_query("""
    SELECT
        COUNT(*) AS Total_Rows,
        COUNT(DISTINCT Bid_Reference) AS Unique_Bids,
        SUM(
            CASE WHEN Bid_Reference IS NULL
            THEN 1 ELSE 0 END
        ) AS Missing_Bid_References
    FROM bid_audit
""", con)

print("\nBID-LEVEL SUMMARY")
display(summary)

# Find references occurring in multiple quarters
cross_quarter = pd.read_sql_query("""
    SELECT COUNT(*) AS Bids_In_Multiple_Quarters
    FROM (
        SELECT Bid_Reference
        FROM bid_audit
        WHERE Bid_Reference IS NOT NULL
        GROUP BY Bid_Reference
        HAVING COUNT(DISTINCT Source_Quarter) > 1
    )
""", con)

print("\nCROSS-QUARTER REFERENCES")
display(cross_quarter)

# Identify bids marked both Awarded and Failed
conflicts = pd.read_sql_query("""
    SELECT COUNT(*) AS Awarded_And_Failed_References
    FROM (
        SELECT Bid_Reference
        FROM bid_audit
        WHERE Bid_Reference IS NOT NULL
        GROUP BY Bid_Reference
        HAVING
            SUM(CASE WHEN Bid_Status = 'awarded'
                THEN 1 ELSE 0 END) > 0
        AND
            SUM(CASE WHEN Bid_Status = 'failed'
                THEN 1 ELSE 0 END) > 0
    )
""", con)

print("\nCONFLICTING OUTCOMES")
display(conflicts)

# Status distribution by quarter
status_report = pd.read_sql_query("""
    SELECT
        Source_Quarter,
        COALESCE(Bid_Status, 'missing') AS Bid_Status,
        COUNT(*) AS Records
    FROM bid_audit
    GROUP BY Source_Quarter, Bid_Status
    ORDER BY Source_Quarter, Bid_Status
""", con)

display(
    status_report.pivot(
        index="Source_Quarter",
        columns="Bid_Status",
        values="Records"
    ).fillna(0).astype(int)
)

con.close()


### 5.2 Define the target conservatively

A bid is **Awarded (`Target_Success = 1`)** only when it has one consistent status and that status is `awarded`. It is **Failed (`Target_Success = 0`)** only when the consistent status is `failed`. Other, missing or conflicting statuses remain **unresolved (`NULL`)** and are excluded from supervised modeling.

The saved results show **441,738 Awarded**, **59,733 Failed**, and **553,971 unresolved** unique bids before additional feature-quality exclusions. A single multi-status cross-quarter reference is also flagged. In subsequent model files, the positive outcome is inverted to **`Target_Failed = 1`**.

In [ ]:

import sqlite3
import pandas as pd

con = sqlite3.connect(db_path)

# Rebuild bid-level outcome table
con.execute("DROP TABLE IF EXISTS bid_level_outcomes")

con.execute("""
CREATE TABLE bid_level_outcomes AS

SELECT
    Bid_Reference,

    MIN(Source_Quarter) AS Source_Quarter,

    COUNT(*) AS Item_Rows,

    COUNT(DISTINCT Source_Quarter) AS Quarter_Count,

    COUNT(
        DISTINCT COALESCE(Bid_Status, 'missing')
    ) AS Status_Count,

    GROUP_CONCAT(
        DISTINCT Bid_Status
    ) AS Status_Values,

    CASE
        WHEN COUNT(
            DISTINCT COALESCE(Bid_Status, 'missing')
        ) = 1
        AND MIN(Bid_Status) = 'awarded'
        THEN 1

        WHEN COUNT(
            DISTINCT COALESCE(Bid_Status, 'missing')
        ) = 1
        AND MIN(Bid_Status) = 'failed'
        THEN 0

        ELSE NULL
    END AS Target_Success

FROM bid_audit

WHERE Bid_Reference IS NOT NULL

GROUP BY Bid_Reference
""")

con.commit()

# Overall bid-level summary
summary = pd.read_sql_query("""
SELECT
    COUNT(*) AS Total_Unique_Bids,

    SUM(CASE WHEN Target_Success = 1
        THEN 1 ELSE 0 END) AS Awarded_Bids,

    SUM(CASE WHEN Target_Success = 0
        THEN 1 ELSE 0 END) AS Failed_Bids,

    SUM(CASE WHEN Target_Success IS NULL
        THEN 1 ELSE 0 END) AS Unresolved_Bids,

    SUM(CASE WHEN Status_Count > 1
        THEN 1 ELSE 0 END) AS Multiple_Status_Bids

FROM bid_level_outcomes
""", con)

print("FULL BID-LEVEL SUMMARY")
display(summary)

# Outcome distribution by quarter
quarterly = pd.read_sql_query("""
SELECT
    Source_Quarter,
    COUNT(*) AS Unique_Bids,

    SUM(CASE WHEN Target_Success = 1
        THEN 1 ELSE 0 END) AS Awarded,

    SUM(CASE WHEN Target_Success = 0
        THEN 1 ELSE 0 END) AS Failed,

    SUM(CASE WHEN Target_Success IS NULL
        THEN 1 ELSE 0 END) AS Unresolved

FROM bid_level_outcomes

GROUP BY Source_Quarter
ORDER BY Source_Quarter
""", con)

print("\nQUARTERLY BID-LEVEL OUTCOMES")
display(quarterly)

con.close()


### 5.3 First cross-quarter EDA: outcomes and unresolved coverage

These plots summarize labeled Awarded/Failed bids by quarter and identify exceptional references. **The rate among labeled bids is conditional on having a resolved label**; it is not the verified failure rate for all published procurement notices. Later sections revisit this bias, especially for newer periods.

In [ ]:

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

con = sqlite3.connect(db_path)

# 1. Investigate references with multiple statuses
exceptions = pd.read_sql_query("""
SELECT
    b.Bid_Reference,
    b.Source_Quarter,
    b.Bid_Status,
    COUNT(*) AS Records
FROM bid_audit b
JOIN bid_level_outcomes o
    ON b.Bid_Reference = o.Bid_Reference
WHERE o.Status_Count > 1
GROUP BY
    b.Bid_Reference,
    b.Source_Quarter,
    b.Bid_Status
ORDER BY
    b.Bid_Reference,
    b.Source_Quarter
""", con)

print("BIDS WITH MULTIPLE STATUSES")
display(exceptions)

# 2. Bid-level outcome counts by quarter
eda = pd.read_sql_query("""
SELECT
    Source_Quarter,
    SUM(CASE WHEN Target_Success = 1
        THEN 1 ELSE 0 END) AS Awarded,
    SUM(CASE WHEN Target_Success = 0
        THEN 1 ELSE 0 END) AS Failed,
    SUM(CASE WHEN Target_Success IS NULL
        THEN 1 ELSE 0 END) AS Unresolved
FROM bid_level_outcomes
GROUP BY Source_Quarter
ORDER BY Source_Quarter
""", con)

con.close()

# 3. Calculate outcome percentages
eda["Labeled"] = eda["Awarded"] + eda["Failed"]

eda["Award_Rate_Pct"] = (
    eda["Awarded"] / eda["Labeled"] * 100
).round(2)

print("\nQUARTERLY BID-LEVEL EDA")
display(eda)

# 4. Visualize labeled outcomes
ax = eda.plot(
    x="Source_Quarter",
    y=["Awarded", "Failed"],
    kind="bar",
    figsize=(13, 5)
)

ax.set_title("PhilGEPS: Labeled Public Bidding Outcomes")
ax.set_xlabel("Source Quarter")
ax.set_ylabel("Unique Bid References")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### 5.4 Feature completeness and source consistency

The feature audit measures valid entries for **approved budget, funding source, category, region, publication date, notice title and item name** by quarter. It is followed by sample inspections to distinguish populated text fields from semantically comparable field values. High completeness does **not** guarantee that the meaning of a feature remained consistent across years.

In [ ]:

import pandas as pd
from collections import defaultdict, Counter

source = (
    output_dir.parent
    / "dataset_cleaned"
    / "PHILGEPS_PUBLIC_BIDDING_ALL.csv"
)

features = [
    "Approved_Budget",
    "Funding_Source",
    "Procurement_Category",
    "Region",
    "Published_Date",
    "Notice_Title",
    "Item_Name"
]

required = ["Source_Quarter"] + features
audit = defaultdict(Counter)

null_values = {
    "", "null", "none", "nan",
    "<na>", "n/a", "na"
}

for chunk in pd.read_csv(
    source,
    usecols=required,
    chunksize=50000,
    dtype="string",
    encoding="utf-8-sig"
):
    for quarter, group in chunk.groupby(
        "Source_Quarter", dropna=False
    ):
        audit[quarter]["Rows"] += len(group)

        for col in features:
            values = (
                group[col]
                .fillna("")
                .str.strip()
            )

            valid = ~(
                values.str.casefold().isin(null_values)
            )

            audit[quarter][col] += int(valid.sum())

            if col == "Approved_Budget":
                numeric = pd.to_numeric(
                    values.str.replace(
                        r"[^\d.-]", "", regex=True
                    ),
                    errors="coerce"
                )
                audit[quarter]["Numeric_Budget"] += (
                    int(numeric.notna().sum())
                )

# Prepare completeness report
report = []

for quarter in sorted(audit):
    stats = audit[quarter]
    rows = stats["Rows"]

    record = {
        "Quarter": quarter,
        "Records": rows
    }

    for col in features:
        record[col + "_Pct"] = round(
            stats[col] / rows * 100, 2
        )

    record["Numeric_Budget_Pct"] = round(
        stats["Numeric_Budget"] / rows * 100, 2
    )

    report.append(record)

completeness_df = pd.DataFrame(report)

display(completeness_df)

# Save report
report_path = (
    output_dir.parent
    / "dataset_cleaned"
    / "PHILGEPS_Feature_Completeness.csv"
)

completeness_df.to_csv(
    report_path, index=False
)

print("Saved:", report_path)


In [ ]:

import pandas as pd

source = (
    output_dir.parent
    / "dataset_cleaned"
    / "PHILGEPS_PUBLIC_BIDDING_ALL.csv"
)

quarters_to_check = [
    "2023_Q1",
    "2024_Q4",
    "2025_Q3",
    "2025_Q4",
    "2026_Q1",
    "2026_Q2"
]

columns = [
    "Source_Quarter",
    "Approved_Budget",
    "Funding_Source",
    "Procurement_Category",
    "Region",
    "Published_Date",
    "Item_Name"
]

samples = {}

for chunk in pd.read_csv(
    source,
    usecols=columns,
    chunksize=50000,
    dtype="string",
    encoding="utf-8-sig"
):

    for quarter in quarters_to_check:

        if quarter in samples:
            continue

        selected = chunk.loc[
            chunk["Source_Quarter"].eq(quarter)
        ]

        if not selected.empty:
            samples[quarter] = selected.head(5)

    if len(samples) == len(quarters_to_check):
        break

for quarter in quarters_to_check:
    print("\n", "=" * 50)
    print("QUARTER:", quarter)

    if quarter in samples:
        display(samples[quarter])
    else:
        print("No records found.")

# Review previously calculated budget validation
print("\nNUMERIC BUDGET COMPLETENESS")

display(
    completeness_df[
        ["Quarter", "Approved_Budget_Pct",
         "Numeric_Budget_Pct", "Item_Name_Pct"]
    ]
)


### 5.5 Date and budget validation

Publication dates are parsed with the source-specific date format (`DD/MM/YYYY` in 2023–2025; `DD-Mon-YYYY` in 2026), then checked against the reported source quarter. Approved budgets are validated as numeric and checked for zero or negative values. These tests support using publication timing and positive approved budget as pre-outcome predictors.

In [ ]:

import pandas as pd
from collections import defaultdict

source = (
    output_dir.parent
    / "dataset_cleaned"
    / "PHILGEPS_PUBLIC_BIDDING_ALL.csv"
)

audit = defaultdict(lambda: {
    "Rows": 0,
    "Valid_Dates": 0,
    "Invalid_Dates": 0,
    "Outside_Quarter": 0,
    "Valid_Budgets": 0,
    "Zero_Negative_Budgets": 0
})

for chunk in pd.read_csv(
    source,
    usecols=[
        "Source_Quarter",
        "Published_Date",
        "Approved_Budget"
    ],
    dtype="string",
    chunksize=50000,
    encoding="utf-8-sig"
):

    for quarter, group in chunk.groupby("Source_Quarter"):

        year = int(quarter[:4])
        quarter_no = int(quarter[-1])

        # Use format appropriate to source period
        if year <= 2025:
            date_format = "%d/%m/%Y"
        else:
            date_format = "%d-%b-%Y"

        dates = pd.to_datetime(
            group["Published_Date"].str.strip(),
            format=date_format,
            errors="coerce"
        )

        budget = pd.to_numeric(
            group["Approved_Budget"]
            .str.replace(",", "", regex=False)
            .str.strip(),
            errors="coerce"
        )

        valid_dates = dates.notna()
        valid_budgets = budget.notna()

        outside_quarter = (
            valid_dates
            & (
                (dates.dt.year != year)
                | (dates.dt.quarter != quarter_no)
            )
        )

        stats = audit[quarter]

        stats["Rows"] += len(group)
        stats["Valid_Dates"] += int(valid_dates.sum())
        stats["Invalid_Dates"] += int((~valid_dates).sum())
        stats["Outside_Quarter"] += int(
            outside_quarter.sum()
        )
        stats["Valid_Budgets"] += int(
            valid_budgets.sum()
        )
        stats["Zero_Negative_Budgets"] += int(
            (valid_budgets & (budget <= 0)).sum()
        )

# Create audit report
date_budget_report = pd.DataFrame([
    {"Quarter": quarter, **values}
    for quarter, values in sorted(audit.items())
])

date_budget_report["Date_Valid_Pct"] = (
    date_budget_report["Valid_Dates"]
    / date_budget_report["Rows"] * 100
).round(2)

date_budget_report["Budget_Valid_Pct"] = (
    date_budget_report["Valid_Budgets"]
    / date_budget_report["Rows"] * 100
).round(2)

display(date_budget_report)

# Save audit
report_file = (
    output_dir.parent
    / "dataset_cleaned"
    / "PHILGEPS_Date_Budget_Audit.csv"
)

date_budget_report.to_csv(
    report_file, index=False
)

print("Total invalid dates:",
      date_budget_report["Invalid_Dates"].sum())

print("Dates outside source quarter:",
      date_budget_report["Outside_Quarter"].sum())

print("Zero/negative budgets:",
      date_budget_report["Zero_Negative_Budgets"].sum())

print("\nAudit saved to:", report_file)


### 5.6 Aggregate input features to bid level

A second SQLite stage calculates a single bid-level feature record, including minimum/maximum approved budget, earliest publication date, funding source, category and region. **Version counts** flag inconsistent values within a reference. Because the modeling table must have one trustworthy row per bid, ambiguous or nonpositive budgets and inconsistent feature values are reviewed and, where specified below, excluded.

In [ ]:

import pandas as pd
import sqlite3
from pathlib import Path

clean_dir = output_dir.parent / "dataset_cleaned"
source = clean_dir / "PHILGEPS_PUBLIC_BIDDING_ALL.csv"
db_path = clean_dir / "PHILGEPS_BID_AUDIT.sqlite"

con = sqlite3.connect(db_path)

con.execute("DROP TABLE IF EXISTS bid_feature_stage")
con.execute("DROP TABLE IF EXISTS bid_level_features")

fields = [
    "Bid_Reference",
    "Approved_Budget",
    "Published_Date",
    "Source_Quarter",
    "Funding_Source",
    "Procurement_Category",
    "Region"
]

total = 0

for chunk in pd.read_csv(
    source,
    usecols=fields,
    chunksize=50000,
    dtype="string",
    encoding="utf-8-sig"
):

    chunk["Bid_Reference"] = (
        chunk["Bid_Reference"].str.strip()
    )

    chunk["Approved_Budget"] = pd.to_numeric(
        chunk["Approved_Budget"]
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

    # Standardize publication dates
    is_2026 = chunk["Source_Quarter"].str.startswith(
        "2026", na=False
    )

    dates = pd.Series(pd.NaT, index=chunk.index)

    dates.loc[~is_2026] = pd.to_datetime(
        chunk.loc[~is_2026, "Published_Date"],
        format="%d/%m/%Y",
        errors="coerce"
    )

    dates.loc[is_2026] = pd.to_datetime(
        chunk.loc[is_2026, "Published_Date"],
        format="%d-%b-%Y",
        errors="coerce"
    )

    chunk["Published_Date"] = (
        dates.dt.strftime("%Y-%m-%d")
    )

    for col in [
        "Funding_Source",
        "Procurement_Category",
        "Region"
    ]:
        chunk[col] = chunk[col].str.strip()

    chunk.to_sql(
        "bid_feature_stage",
        con,
        if_exists="append",
        index=False
    )

    total += len(chunk)

print("Rows staged:", f"{total:,}")

# Create one observation per bid reference
con.execute("""
CREATE TABLE bid_level_features AS
SELECT
    s.Bid_Reference,

    COUNT(*) AS Item_Rows,

    COUNT(DISTINCT s.Approved_Budget)
        AS Budget_Versions,

    MIN(s.Approved_Budget)
        AS Minimum_Budget,

    MAX(s.Approved_Budget)
        AS Maximum_Budget,

    COUNT(DISTINCT s.Published_Date)
        AS Date_Versions,

    MIN(s.Published_Date)
        AS Published_Date,

    COUNT(DISTINCT s.Funding_Source)
        AS Funding_Versions,

    MIN(s.Funding_Source)
        AS Funding_Source,

    COUNT(DISTINCT s.Procurement_Category)
        AS Category_Versions,

    MIN(s.Procurement_Category)
        AS Procurement_Category,

    COUNT(DISTINCT s.Region)
        AS Region_Versions,

    MIN(s.Region)
        AS Region

FROM bid_feature_stage s
WHERE s.Bid_Reference IS NOT NULL
GROUP BY s.Bid_Reference
""")

con.commit()

# Audit potential issues
audit_summary = pd.read_sql_query("""
SELECT
    COUNT(*) AS Total_Bids,

    SUM(CASE WHEN Budget_Versions > 1
        THEN 1 ELSE 0 END)
        AS Inconsistent_Budgets,

    SUM(CASE WHEN Minimum_Budget <= 0
        THEN 1 ELSE 0 END)
        AS Nonpositive_Budget_Bids,

    SUM(CASE WHEN Date_Versions > 1
        THEN 1 ELSE 0 END)
        AS Inconsistent_Dates,

    SUM(CASE WHEN Funding_Versions > 1
        THEN 1 ELSE 0 END)
        AS Multiple_Funding_Sources,

    SUM(CASE WHEN Category_Versions > 1
        THEN 1 ELSE 0 END)
        AS Multiple_Categories,

    SUM(CASE WHEN Region_Versions > 1
        THEN 1 ELSE 0 END)
        AS Multiple_Regions

FROM bid_level_features
""", con)

print("\nBID-LEVEL FEATURE VALIDATION")
display(audit_summary)

print("\nSAMPLE BIDS REQUIRING BUDGET REVIEW")

budget_exceptions = pd.read_sql_query("""
SELECT
    Bid_Reference,
    Item_Rows,
    Budget_Versions,
    Minimum_Budget,
    Maximum_Budget

FROM bid_level_features

WHERE Budget_Versions > 1
   OR Minimum_Budget <= 0
   OR Budget_Versions = 0

ORDER BY Item_Rows DESC
LIMIT 15
""", con)

display(budget_exceptions)

con.close()


In [ ]:

import sqlite3
import pandas as pd

con = sqlite3.connect(db_path)

tables = pd.read_sql_query("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name
""", con)

display(tables)

if "bid_level_features" in tables["name"].values:
    summary = pd.read_sql_query("""
    SELECT
        COUNT(*) AS Total_Bids,
        SUM(CASE WHEN Budget_Versions > 1 THEN 1 ELSE 0 END)
            AS Inconsistent_Budgets,
        SUM(CASE WHEN Minimum_Budget <= 0 THEN 1 ELSE 0 END)
            AS Nonpositive_Budgets,
        SUM(CASE WHEN Date_Versions > 1 THEN 1 ELSE 0 END)
            AS Inconsistent_Dates,
        SUM(CASE WHEN Funding_Versions > 1 THEN 1 ELSE 0 END)
            AS Multiple_Funding_Sources,
        SUM(CASE WHEN Category_Versions > 1 THEN 1 ELSE 0 END)
            AS Multiple_Categories,
        SUM(CASE WHEN Region_Versions > 1 THEN 1 ELSE 0 END)
            AS Multiple_Regions
    FROM bid_level_features
    """, con)

    display(summary)
else:
    print("Bid-level aggregation has not completed.")

con.close()


### 5.7 Build the clean modeling table and exception log

The two bid-level tables are joined. Only resolved outcomes are eligible; records with invalid, missing, nonpositive or inconsistent key fields are written separately to an exceptions file. Remaining rows receive `Target_Failed`, `Log_Approved_Budget = log(1 + budget)`, publication year and month.

**Final modeling dataset (saved output):** **501,298 accepted bids** from **501,471 labeled bids**, with **173 excluded for data quality**. This is the dataset used for the modeling and the later EDA. Exclusion rules can limit representativeness and should be documented in any presentation.

In [ ]:

import pandas as pd
import numpy as np
import sqlite3

clean_dir = output_dir.parent / "dataset_cleaned"

model_file = clean_dir / "PHILGEPS_BID_LEVEL_MODEL.csv"
exception_file = clean_dir / "PHILGEPS_MODEL_EXCEPTIONS.csv"

con = sqlite3.connect(db_path)

query = """
SELECT
    o.Bid_Reference,
    o.Source_Quarter,
    o.Target_Success,
    o.Status_Count,

    f.Published_Date,
    f.Minimum_Budget AS Approved_Budget,
    f.Funding_Source,
    f.Procurement_Category,
    f.Region,

    f.Budget_Versions,
    f.Date_Versions,
    f.Funding_Versions,
    f.Category_Versions,
    f.Region_Versions

FROM bid_level_outcomes o

INNER JOIN bid_level_features f
    ON o.Bid_Reference = f.Bid_Reference

WHERE o.Target_Success IN (0, 1)
"""

first_model = True
first_exception = True

accepted_count = 0
excluded_count = 0

for chunk in pd.read_sql_query(
    query, con, chunksize=50000
):

    chunk["Approved_Budget"] = pd.to_numeric(
        chunk["Approved_Budget"],
        errors="coerce"
    )

    chunk["Published_Date"] = pd.to_datetime(
        chunk["Published_Date"],
        errors="coerce"
    )

    # Flag invalid or inconsistent records
    budget_issue = (
        (chunk["Budget_Versions"] != 1)
        | chunk["Approved_Budget"].isna()
        | (chunk["Approved_Budget"] <= 0)
    )

    date_issue = (
        (chunk["Date_Versions"] != 1)
        | chunk["Published_Date"].isna()
    )

    funding_issue = chunk["Funding_Versions"] != 1
    category_issue = chunk["Category_Versions"] != 1
    region_issue = chunk["Region_Versions"] != 1

    # Normalize placeholder text values
    for col in [
        "Funding_Source",
        "Procurement_Category",
        "Region"
    ]:
        chunk[col] = (
            chunk[col]
            .astype("string")
            .str.strip()
        )

        chunk[col] = chunk[col].mask(
            chunk[col].str.casefold().isin(
                ["", "null", "none", "nan", "n/a", "<na>"]
            )
        )

    missing_category = chunk[
        ["Funding_Source", "Procurement_Category", "Region"]
    ].isna().any(axis=1)

    status_issue = chunk["Status_Count"] != 1

    exclude = (
        budget_issue
        | date_issue
        | funding_issue
        | category_issue
        | region_issue
        | missing_category
        | status_issue
    )

    # Save exceptions for audit
    exceptions = chunk.loc[exclude].copy()

    if not exceptions.empty:
        exceptions.to_csv(
            exception_file,
            mode="w" if first_exception else "a",
            header=first_exception,
            index=False,
            encoding="utf-8-sig"
        )
        first_exception = False

    excluded_count += len(exceptions)

    # Prepare clean modeling observations
    model = chunk.loc[~exclude].copy()

    if model.empty:
        continue

    # Positive class represents failed bidding
    model["Target_Failed"] = (
        1 - model["Target_Success"].astype(int)
    )

    model["Log_Approved_Budget"] = np.log1p(
        model["Approved_Budget"]
    )

    model["Publication_Year"] = (
        model["Published_Date"].dt.year
    )

    model["Publication_Month"] = (
        model["Published_Date"].dt.month
    )

    final_columns = [
        "Bid_Reference",
        "Source_Quarter",
        "Published_Date",
        "Approved_Budget",
        "Log_Approved_Budget",
        "Funding_Source",
        "Procurement_Category",
        "Region",
        "Publication_Year",
        "Publication_Month",
        "Target_Failed"
    ]

    model[final_columns].to_csv(
        model_file,
        mode="w" if first_model else "a",
        header=first_model,
        index=False,
        encoding="utf-8-sig"
    )

    first_model = False
    accepted_count += len(model)

con.close()

print("MODEL DATASET CREATED")
print("Accepted bids:", f"{accepted_count:,}")
print("Excluded bids:", f"{excluded_count:,}")
print("Total labeled bids:", f"{accepted_count + excluded_count:,}")

print("\nSaved:", model_file)
print("Exception file created:", not first_exception)


### 5.8 Inspect the final sample and class balance

Before fitting any algorithm, the notebook checks year counts, distributions and the target variable. **`Target_Failed = 1` is the positive class**, so Precision and Recall later refer to detecting failed bidding, not predicting winners.

In [ ]:

# Read the output, which is smaller than the
# original item-level dataset

ml_df = pd.read_csv(model_file)

print("Total modeling observations:", len(ml_df))

print("\nTarget distribution:")
display(
    ml_df["Target_Failed"]
    .value_counts()
)

print("\nDistribution by year:")
display(
    pd.crosstab(
        ml_df["Publication_Year"],
        ml_df["Target_Failed"],
        margins=True
    )
)

print("\nDuplicate bid references:")
print(ml_df["Bid_Reference"].duplicated().sum())

print("\nSample records:")
display(ml_df.head())


## Phase 6. Model development and 2025 validation

### 6.1 Time-based split, features, preprocessing and baseline

The analysis uses a chronological split to simulate prediction on future periods: **2023–2024 for training**, **2025 for validation**, and **2026 Q1–Q2 as an untouched, later-period diagnostic**. This is preferable to randomly mixing the same reporting periods when evaluating temporal generalization.

Input features are **log approved budget, funding source, procurement category, region and publication month**. Numeric fields are standardized, and categorical fields one-hot encoded. A majority-class baseline and class-weighted logistic regression establish reference performance.

**Recorded sample sizes:** 387,488 training, 105,333 validation and 8,477 2026 diagnostic bids. The 2025 baseline accuracy is **88.5%**, but it detects **0% of failures**, illustrating why accuracy by itself is insufficient for this imbalanced task.

In [ ]:

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Load bid-level dataset
ml_df = pd.read_csv(
    model_file,
    dtype={"Bid_Reference": "string"}
)

# Chronological split
train = ml_df[
    ml_df["Publication_Year"].isin([2023, 2024])
].copy()

valid = ml_df[
    ml_df["Publication_Year"] == 2025
].copy()

holdout = ml_df[
    ml_df["Publication_Year"] == 2026
].copy()

features = [
    "Log_Approved_Budget",
    "Funding_Source",
    "Procurement_Category",
    "Region",
    "Publication_Month"
]

categorical = [
    "Funding_Source",
    "Procurement_Category",
    "Region"
]

numeric = [
    "Log_Approved_Budget",
    "Publication_Month"
]

X_train = train[features].copy()
X_valid = valid[features].copy()

for col in categorical:
    X_train[col] = X_train[col].fillna("Unknown")
    X_valid[col] = X_valid[col].fillna("Unknown")

y_train = train["Target_Failed"].astype(int)
y_valid = valid["Target_Failed"].astype(int)

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric),
    ("categorical",
     OneHotEncoder(
         handle_unknown="ignore",
         min_frequency=100
     ), categorical)
])

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        class_weight="balanced",
        solver="liblinear",
        max_iter=500,
        random_state=42
    ))
])

# Train only on 2023 and 2024
model.fit(X_train, y_train)

# Predict 2025
pred = model.predict(X_valid)
proba = model.predict_proba(X_valid)[:, 1]

# Majority-class baseline
baseline = np.zeros(len(y_valid), dtype=int)

print("TRAINING:", len(train))
print("VALIDATION:", len(valid))
print("2026 HOLDOUT:", len(holdout))

print("\nMAJORITY-CLASS BASELINE")
print("Accuracy:", round(
    accuracy_score(y_valid, baseline), 4
))
print("Failed recall:", round(
    recall_score(y_valid, baseline), 4
))

print("\nLOGISTIC REGRESSION RESULTS")
print(classification_report(
    y_valid, pred,
    target_names=["Awarded", "Failed"],
    zero_division=0
))

print("ROC-AUC:", round(
    roc_auc_score(y_valid, proba), 4
))

print("PR-AUC:", round(
    average_precision_score(y_valid, proba), 4
))

print("\nCONFUSION MATRIX")
display(pd.DataFrame(
    confusion_matrix(y_valid, pred),
    index=["Actual Awarded", "Actual Failed"],
    columns=["Predicted Awarded", "Predicted Failed"]
))


### 6.2 Translate model scores into a review queue

Instead of using a single arbitrary probability threshold, rank bids from highest to lowest **predicted failure risk**. At fixed review capacities (top 5%, 10%, 20%, 30% or 50%), calculate:

- **Precision@k:** actual failed bids among the reviewed top-k%.
- **Recall@k:** fraction of *all labeled failures* included in that top-k%.
- **Lift@k:** Precision@k divided by the period's overall failure rate.

These metrics answer a management question: **If staff can review only a limited share of opportunities, how many likely failures can that queue surface?** They are more operationally meaningful than accuracy alone.

In [ ]:

import numpy as np
import pandas as pd

# Use existing 2025 validation predictions
risk_df = pd.DataFrame({
    "Actual_Failed": np.asarray(y_valid),
    "Risk_Score": np.asarray(proba)
})

risk_df = risk_df.sort_values(
    "Risk_Score", ascending=False
).reset_index(drop=True)

total_failures = risk_df["Actual_Failed"].sum()
overall_failure_rate = risk_df["Actual_Failed"].mean()

results = []

for review_share in [0.05, 0.10, 0.20, 0.30, 0.50]:

    n_review = int(np.ceil(
        len(risk_df) * review_share
    ))

    selected = risk_df.iloc[:n_review]

    detected = int(selected["Actual_Failed"].sum())

    precision = detected / n_review
    recall = detected / total_failures
    lift = precision / overall_failure_rate

    results.append({
        "Review_Share": f"{review_share:.0%}",
        "Bids_Flagged": n_review,
        "Actual_Failures_Found": detected,
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "Lift_vs_Random": round(lift, 2)
    })

priority_results = pd.DataFrame(results)

display(priority_results)


### 6.3 Compare Logistic Regression, Decision Tree and Random Forest

Each model uses the same 2023–2024 training and 2025 validation split. The comparison reports **ROC-AUC**, **PR-AUC (average precision)** and review-capacity metrics. The Random Forest performs best among these tested specifications on 2025 validation, but the purpose of this notebook is to describe measured results and limitations, **not endlessly tune for a higher score**.

**2025 Random Forest:** ROC-AUC **0.726**, PR-AUC **0.343**; among the top **10%** risk-ranked bids, **36.4% precision**, **31.7% recall** and **3.17× lift**. This is a ranking result on the *labeled 2025 population*, not a demonstrated intervention effect.

In [ ]:

from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
import numpy as np
import pandas as pd

# Existing Logistic Regression scores
models = {
    "Logistic Regression": model,

    "Decision Tree": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", DecisionTreeClassifier(
            max_depth=10,
            min_samples_leaf=100,
            class_weight="balanced",
            random_state=42
        ))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=100,
            max_depth=14,
            min_samples_leaf=60,
            class_weight="balanced_subsample",
            n_jobs=-1,
            random_state=42
        ))
    ])
}

results = []
y_true = np.asarray(y_valid)
total_failures = y_true.sum()
base_rate = y_true.mean()

for name, clf in models.items():

    if name != "Logistic Regression":
        clf.fit(X_train, y_train)

    scores = clf.predict_proba(X_valid)[:, 1]
    ranked = np.argsort(-scores, kind="stable")

    result = {
        "Model": name,
        "ROC_AUC": roc_auc_score(y_true, scores),
        "PR_AUC": average_precision_score(
            y_true, scores
        )
    }

    for share in [0.05, 0.10, 0.20]:

        n = int(np.ceil(len(y_true) * share))
        selected = y_true[ranked[:n]]

        precision = selected.mean()
        recall = selected.sum() / total_failures
        lift = precision / base_rate

        label = f"{int(share * 100)}pct"

        result[f"Precision_{label}"] = precision
        result[f"Recall_{label}"] = recall
        result[f"Lift_{label}"] = lift

    results.append(result)

comparison_df = pd.DataFrame(results)

display(
    comparison_df.round(3)
)

# Save results for final project documentation
comparison_df.to_csv(
    clean_dir / "PHILGEPS_Model_Comparison.csv",
    index=False
)


## Phase 7. Out-of-time robustness and data drift

### 7.1 Evaluate the original Random Forest on 2026

The previously fitted 2023–2024 model is applied without retraining to 2026 Q1–Q2, testing how well its ranking transfers to a later period. The recorded performance weakens: **ROC-AUC 0.574** and **top-10% lift 1.55×**. Precision at a fixed 0.50 threshold is especially hard to interpret when probability calibration and prevalence shift, so ranking diagnostics are reported separately.

**Interpretation:** Earlier-period validation does **not** demonstrate that the model is deployment-ready in 2026.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# Use previously fitted Random Forest
rf_model = models["Random Forest"]

X_test = holdout[features].copy()
y_test = holdout["Target_Failed"].astype(int)

for col in categorical:
    X_test[col] = X_test[col].fillna("Unknown")

# Predict with the existing model
test_scores = rf_model.predict_proba(X_test)[:, 1]

test_pred = (test_scores >= 0.50).astype(int)

print("2026 HOLDOUT EVALUATION")
print("Total bids:", len(y_test))
print("Actual failure rate:", round(y_test.mean(), 4))

print("\nMODEL METRICS")
print("ROC-AUC:", round(
    roc_auc_score(y_test, test_scores), 4
))
print("PR-AUC:", round(
    average_precision_score(y_test, test_scores), 4
))
print("Failed Precision:", round(
    precision_score(y_test, test_pred, zero_division=0), 4
))
print("Failed Recall:", round(
    recall_score(y_test, test_pred, zero_division=0), 4
))
print("Failed F1:", round(
    f1_score(y_test, test_pred, zero_division=0), 4
))

# Risk-prioritization assessment
ranked = np.argsort(-test_scores, kind="stable")
results = []

for share in [0.05, 0.10, 0.20]:
    n = int(np.ceil(len(y_test) * share))

    selected = y_test.iloc[ranked[:n]]

    results.append({
        "Review_Capacity": f"{share:.0%}",
        "Bids_Reviewed": n,
        "Precision": selected.mean(),
        "Recall": selected.sum() / y_test.sum(),
        "Lift": selected.mean() / y_test.mean()
    })

holdout_results = pd.DataFrame(results)

print("\n2026 RISK PRIORITIZATION")
display(holdout_results.round(3))


### 7.2 Diagnose the performance drop

The original model flags about **97.3%** of 2026 labeled bids as failed at a 0.50 threshold, versus a **38.4%** observed failure rate in the labeled 2026 subset. The feature-level diagnosis shows:

- **100%** of 2026 funding-source values are unseen in training.
- **55.5%** of 2026 procurement-category labels are unseen in training.
- Region labels have a much smaller unseen rate.

These are signs of **reporting/schema drift**, compounded by shifts in observed outcomes and in the population that receives final labels. They explain why the next experiment tests harmonized categories without pretending it resolves all validity concerns.

In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix

# Create 2026 diagnostic dataset
diagnostic = holdout[
    ["Source_Quarter", "Funding_Source",
     "Procurement_Category", "Region"]
].copy()

diagnostic["Actual_Failed"] = np.asarray(y_test)
diagnostic["Risk_Score"] = test_scores
diagnostic["Predicted_Failed"] = (
    test_scores >= 0.50
).astype(int)

# A. Check prediction behavior
print("A. PREDICTION DISTRIBUTION")

print("Actual failure rate:",
      round(diagnostic["Actual_Failed"].mean(), 3))

print("Predicted failure rate:",
      round(diagnostic["Predicted_Failed"].mean(), 3))

print("\nCONFUSION MATRIX")
display(pd.DataFrame(
    confusion_matrix(
        diagnostic["Actual_Failed"],
        diagnostic["Predicted_Failed"]
    ),
    index=["Actual Awarded", "Actual Failed"],
    columns=["Predicted Awarded", "Predicted Failed"]
))

# B. Examine performance by quarter
print("\nB. 2026 QUARTERLY DIAGNOSTICS")

quarter_report = diagnostic.groupby(
    "Source_Quarter"
).agg(
    Bids=("Actual_Failed", "size"),
    Actual_Failure_Rate=("Actual_Failed", "mean"),
    Predicted_Failure_Rate=("Predicted_Failed", "mean"),
    Average_Risk_Score=("Risk_Score", "mean")
)

display(quarter_report.round(3))

# C. Check categories unseen during training
print("\nC. UNSEEN CATEGORIES IN 2026")

for col in categorical:

    train_values = set(
        X_train[col]
        .astype("string")
        .fillna("Unknown")
        .str.strip()
    )

    test_values = (
        X_test[col]
        .astype("string")
        .fillna("Unknown")
        .str.strip()
    )

    unseen = ~test_values.isin(train_values)

    print(
        f"{col}: {unseen.mean():.1%} "
        "of 2026 records contain unseen categories"
    )

    display(
        test_values[unseen]
        .value_counts()
        .head(10)
    )


### 7.3 Harmonization sensitivity test (not the original benchmark)

The follow-up experiment groups procurement categories into **Civil Works, Goods, Consulting Services** (or Other/Unmapped), standardizes region labels, and **temporarily excludes Funding Source** because its 2026 codes are not comparable to the earlier values. It retrains a Random Forest on the same 2023–2024 period.

The harmonized version records **2025 ROC-AUC 0.725** and **2026 ROC-AUC 0.600**, with **2026 top-10% lift 1.57×**. This is a modest robustness improvement over the unharmonized 2026 result, **not** evidence that reporting drift or label bias has been eliminated.

In [ ]:

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

# Work on a copy of the modeling dataset
df2 = ml_df.copy()

# 1. Harmonize procurement categories
category = (
    df2["Procurement_Category"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.casefold()
)

df2["Category_Group"] = np.select(
    [
        category.str.contains("civil works|infra", regex=True),
        category.str.contains("goods", regex=False),
        category.str.contains("consult", regex=False)
    ],
    [
        "Civil Works",
        "Goods",
        "Consulting Services"
    ],
    default="Other / Unmapped"
)

# 2. Standardize region labels
df2["Region_Group"] = (
    df2["Region"]
    .astype("string")
    .fillna("Unknown")
    .str.strip()
    .str.upper()
)

# Check category mapping by year
print("CATEGORY DISTRIBUTION BY YEAR")
display(
    pd.crosstab(
        df2["Publication_Year"],
        df2["Category_Group"],
        normalize="index"
    ).round(3)
)

# 3. Exclude Funding Source temporarily
model_features = [
    "Log_Approved_Budget",
    "Category_Group",
    "Region_Group",
    "Publication_Month"
]

numeric_features = [
    "Log_Approved_Budget",
    "Publication_Month"
]

categorical_features = [
    "Category_Group",
    "Region_Group"
]

# 4. Keep the original time-based split
train2 = df2[df2["Publication_Year"].isin([2023, 2024])]
valid2 = df2[df2["Publication_Year"] == 2025]
diag2026 = df2[df2["Publication_Year"] == 2026]

prep2 = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=100
    ), categorical_features)
])

rf_harmonized = Pipeline([
    ("preprocessor", prep2),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=14,
        min_samples_leaf=60,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    ))
])

# Train using only 2023-2024
rf_harmonized.fit(
    train2[model_features],
    train2["Target_Failed"].astype(int)
)

# 5. Evaluate both periods
results = []

for period, dataset in [
    ("2025 Validation", valid2),
    ("2026 Drift Diagnostic", diag2026)
]:
    X = dataset[model_features]
    y = dataset["Target_Failed"].astype(int).to_numpy()

    scores = rf_harmonized.predict_proba(X)[:, 1]

    ranked = np.argsort(-scores, kind="stable")
    n = int(np.ceil(len(y) * 0.10))
    top10 = y[ranked[:n]]

    precision10 = top10.mean()
    recall10 = top10.sum() / y.sum()

    results.append({
        "Period": period,
        "Bids": len(y),
        "Actual_Failure_Rate": y.mean(),
        "ROC_AUC": roc_auc_score(y, scores),
        "PR_AUC": average_precision_score(y, scores),
        "Predicted_Failed_Pct": (
            (scores >= 0.50).mean()
        ),
        "Precision_Top10": precision10,
        "Recall_Top10": recall10,
        "Lift_Top10": precision10 / y.mean()
    })

harmonized_results = pd.DataFrame(results)

print("\nHARMONIZED RANDOM FOREST RESULTS")
display(harmonized_results.round(3))

# Save experimental results
harmonized_results.to_csv(
    clean_dir / "PHILGEPS_Harmonized_RF_Diagnostic.csv",
    index=False
)


### 7.4 Diagnose by procurement category and quarter

The harmonized model is evaluated separately for categories and for quarter-by-category slices. The checks compare base failure rates, ranking quality and top-10% capture across different mixes of bids. They help identify segments where the model is potentially useful, where results are unstable, and where a high precision figure may simply reflect a higher underlying failure rate.

Small groups and single-class slices are omitted from certain metrics, as those metrics would be unreliable or undefined.

In [ ]:

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)
import pandas as pd
import numpy as np

category_results = []

for year in [2025, 2026]:

    subset = df2.loc[
        df2["Publication_Year"] == year
    ].copy()

    subset["Risk_Score"] = (
        rf_harmonized.predict_proba(
            subset[model_features]
        )[:, 1]
    )

    for category, group in subset.groupby(
        "Category_Group"
    ):

        y = group["Target_Failed"].astype(int).to_numpy()
        scores = group["Risk_Score"].to_numpy()

        n = len(group)
        failures = y.sum()

        # Avoid unstable or undefined metrics
        if n < 100 or y.min() == y.max():
            continue

        top_n = int(np.ceil(n * 0.10))
        selected = np.argsort(
            -scores, kind="stable"
        )[:top_n]

        precision10 = y[selected].mean()
        recall10 = y[selected].sum() / failures

        category_results.append({
            "Year": year,
            "Category": category,
            "Bids": n,
            "Failure_Rate": y.mean(),
            "ROC_AUC": roc_auc_score(y, scores),
            "PR_AUC": average_precision_score(
                y, scores
            ),
            "Precision_Top10": precision10,
            "Recall_Top10": recall10,
            "Lift_Top10": precision10 / y.mean()
        })

category_report = pd.DataFrame(category_results)

display(
    category_report.sort_values(
        ["Category", "Year"]
    ).round(3)
)

category_report.to_csv(
    clean_dir / "PHILGEPS_Category_Drift_Analysis.csv",
    index=False
)


In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

diagnostic_df = df2[
    df2["Publication_Year"].isin([2025, 2026])
].copy()

diagnostic_df["Risk_Score"] = (
    rf_harmonized.predict_proba(
        diagnostic_df[model_features]
    )[:, 1]
)

results = []

for (quarter, category), group in diagnostic_df.groupby(
    ["Source_Quarter", "Category_Group"]
):
    y = group["Target_Failed"].astype(int).to_numpy()
    scores = group["Risk_Score"].to_numpy()

    if len(group) < 100 or len(np.unique(y)) < 2:
        continue

    top_n = int(np.ceil(len(group) * 0.10))
    ranked = np.argsort(-scores, kind="stable")
    top_y = y[ranked[:top_n]]

    results.append({
        "Quarter": quarter,
        "Category": category,
        "Bids": len(group),
        "Failure_Rate": y.mean(),
        "ROC_AUC": roc_auc_score(y, scores),
        "Precision_Top10": top_y.mean(),
        "Recall_Top10": top_y.sum() / y.sum(),
        "Lift_Top10": top_y.mean() / y.mean()
    })

quarter_category_report = pd.DataFrame(results)

display(
    quarter_category_report.sort_values(
        ["Category", "Quarter"]
    ).round(3)
)

quarter_category_report.to_csv(
    clean_dir / "PHILGEPS_Quarter_Category_Diagnostic.csv",
    index=False
)


### 7.5 Quantify incomplete labels and outcome-selection bias

Not every published notice has reached, or is represented with, a final Awarded/Failed status. The notebook calculates **label coverage** (`labeled / all unique bids`) by quarter and the failure rate **conditional on being labeled**. Examples from saved outputs: 2025 Q3 label coverage **16.01%**, 2025 Q4 **19.43%** and 2026 Q1 **75.22%**.

Because both the label coverage and the mix of published notices change over time, these rates are **not directly comparable as unconditional procurement failure rates**. Unresolved bids are not assumed to be successful or unsuccessful.

In [ ]:

import sqlite3
import pandas as pd

con = sqlite3.connect(db_path)

label_audit = pd.read_sql_query("""
SELECT
    Source_Quarter,

    COUNT(*) AS Total_Unique_Bids,

    SUM(
        CASE WHEN Target_Success IN (0, 1)
        THEN 1 ELSE 0 END
    ) AS Labeled_Bids,

    SUM(
        CASE WHEN Target_Success = 1
        THEN 1 ELSE 0 END
    ) AS Awarded_Bids,

    SUM(
        CASE WHEN Target_Success = 0
        THEN 1 ELSE 0 END
    ) AS Failed_Bids,

    SUM(
        CASE WHEN Target_Success IS NULL
        THEN 1 ELSE 0 END
    ) AS Unresolved_Bids

FROM bid_level_outcomes
GROUP BY Source_Quarter
ORDER BY Source_Quarter
""", con)

con.close()

label_audit["Label_Coverage_Pct"] = (
    label_audit["Labeled_Bids"]
    / label_audit["Total_Unique_Bids"] * 100
)

label_audit["Failure_Rate_Labeled_Pct"] = (
    label_audit["Failed_Bids"]
    / label_audit["Labeled_Bids"] * 100
)

display(label_audit.round(2))

# Visualization
import matplotlib.pyplot as plt

ax = label_audit.plot(
    x="Source_Quarter",
    y=[
        "Label_Coverage_Pct",
        "Failure_Rate_Labeled_Pct"
    ],
    kind="line",
    marker="o",
    figsize=(12, 5)
)

ax.set_title("PhilGEPS: Outcome Label Coverage vs Failure Rate")
ax.set_xlabel("Publication Quarter")
ax.set_ylabel("Percentage (%)")
ax.set_ylim(0, 100)

plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:

import sqlite3
import pandas as pd

con = sqlite3.connect(db_path)

status_comparison = pd.read_sql_query("""
SELECT
    Source_Quarter,
    COALESCE(Status_Values, 'missing') AS Status,
    COUNT(*) AS Unique_Bids
FROM bid_level_outcomes
WHERE Source_Quarter IN (
    '2025_Q2',
    '2025_Q3',
    '2025_Q4',
    '2026_Q1',
    '2026_Q2'
)
GROUP BY
    Source_Quarter,
    COALESCE(Status_Values, 'missing')
ORDER BY Source_Quarter, Status
""", con)

con.close()

status_table = status_comparison.pivot_table(
    index="Source_Quarter",
    columns="Status",
    values="Unique_Bids",
    fill_value=0
)

status_pct = (
    status_table.div(
        status_table.sum(axis=1), axis=0
    ) * 100
).round(2)

print("BID-LEVEL STATUS DISTRIBUTION (%)")
display(status_pct)

status_pct.to_csv(
    clean_dir / "PHILGEPS_Bid_Status_Composition.csv"
)


## Phase 8. Model interpretation and substantive EDA

### 8.1 Permutation feature importance (original Random Forest)

The next calculation shuffles each input in a stratified 2025 validation sample and measures the decrease in **average precision (PR-AUC)**. In this fitted model, **log approved budget** and **region** have the largest recorded decreases, followed by procurement category. **Importance measures predictive dependence, not a causal relationship**; correlated features and historical reporting differences can affect the ranking.

In [ ]:

import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

# Use the original fitted Random Forest
rf = models["Random Forest"]

# Stratified sample of 2025 validation data
X_sample, _, y_sample, _ = train_test_split(
    X_valid,
    y_valid,
    train_size=12000,
    stratify=y_valid,
    random_state=42
)

# Measure loss of predictive performance
# when individual predictors are shuffled
importance = permutation_importance(
    rf,
    X_sample,
    y_sample,
    scoring="average_precision",
    n_repeats=3,
    random_state=42,
    n_jobs=1
)

importance_df = pd.DataFrame({
    "Feature": X_sample.columns,
    "PR_AUC_Decrease": importance.importances_mean,
    "Std_Dev": importance.importances_std
}).sort_values(
    "PR_AUC_Decrease",
    ascending=False
)

display(importance_df)

# Visualize
ax = importance_df.sort_values(
    "PR_AUC_Decrease"
).plot(
    x="Feature",
    y="PR_AUC_Decrease",
    kind="barh",
    figsize=(9, 5),
    legend=False
)

ax.set_title(
    "Random Forest: Permutation Feature Importance"
)
ax.set_xlabel("Decrease in Average Precision")
ax.set_ylabel("Predictor")

import matplotlib.pyplot as plt
plt.tight_layout()
plt.show()


### 8.2 Explore observed failure rates across budget deciles

The 2025 labeled bids are divided into ten approved-budget groups. Each group's observed failed-bid share is plotted to see whether failure risk differs by procurement size. This analysis is **descriptive**, sensitive to how the labeled subset is selected, and should not imply that changing the budget itself causes success or failure.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Use 2025 validation data
eda_budget = ml_df[
    ml_df["Publication_Year"] == 2025
].copy()

# Divide bids into 10 budget groups
eda_budget["Budget_Decile"] = pd.qcut(
    eda_budget["Approved_Budget"],
    q=10,
    labels=False,
    duplicates="drop"
) + 1

budget_analysis = (
    eda_budget.groupby("Budget_Decile")
    .agg(
        Number_of_Bids=("Target_Failed", "size"),
        Failed_Bids=("Target_Failed", "sum"),
        Failure_Rate=("Target_Failed", "mean"),
        Median_Budget=("Approved_Budget", "median")
    )
    .reset_index()
)

budget_analysis["Failure_Rate_Pct"] = (
    budget_analysis["Failure_Rate"] * 100
)

display(budget_analysis.round(2))

# Plot observed failure rate
plt.figure(figsize=(10, 5))

plt.bar(
    budget_analysis["Budget_Decile"].astype(str),
    budget_analysis["Failure_Rate_Pct"]
)

plt.title(
    "Public Bidding Failure Rate by Approved Budget Decile"
)

plt.xlabel("Approved Budget Decile (1 = Lowest)")
plt.ylabel("Observed Failure Rate (%)")
plt.tight_layout()
plt.show()


### 8.3 Stratify the budget relationship by category

The analysis is repeated for **Goods** and **Civil Works**. It shows why overall budget patterns should not be interpreted without checking differences in procurement type. Budget-decile definitions are based on the full 2025 labeled set in this exploration.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

# Use the harmonized dataset
budget_category = df2.loc[
    df2["Publication_Year"] == 2025
].copy()

# Create budget deciles across 2025 labeled bids
budget_category["Budget_Decile"] = (
    pd.qcut(
        budget_category["Approved_Budget"],
        q=10,
        labels=False,
        duplicates="drop"
    ) + 1
)

# Focus on the two largest categories
budget_category = budget_category[
    budget_category["Category_Group"].isin(
        ["Goods", "Civil Works"]
    )
]

category_analysis = (
    budget_category.groupby(
        ["Category_Group", "Budget_Decile"]
    )
    .agg(
        Bids=("Target_Failed", "size"),
        Failed=("Target_Failed", "sum"),
        Failure_Rate=("Target_Failed", "mean")
    )
    .reset_index()
)

category_analysis["Failure_Rate_Pct"] = (
    category_analysis["Failure_Rate"] * 100
)

print("FAILURE RATES BY BUDGET DECILE AND CATEGORY")
display(category_analysis.round(2))

# Visualization
pivot = category_analysis.pivot(
    index="Budget_Decile",
    columns="Category_Group",
    values="Failure_Rate_Pct"
)

ax = pivot.plot(
    kind="line",
    marker="o",
    figsize=(11, 5)
)

ax.set_title(
    "Bidding Failure Rate by Approved Budget and Category"
)
ax.set_xlabel("Approved Budget Decile")
ax.set_ylabel("Observed Failure Rate (%)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### 8.4 Adjust for procurement category, region and quarter

The final exploratory model uses a binomial generalized linear model with interactions between **budget decile** and **procurement category**, while adjusting for **region** and **source quarter**.

**Notebook history note:** The next cell is an earlier cleanup step that assumes `adjusted_df` already exists. The following model-building cell recreates `adjusted_df` but contains a saved `statsmodels` pandas-string-dtype error. The last two cells show the successful adjusted fit and predictions from the earlier, prepared in-memory data. For a clean independent rerun, construct `adjusted_df` before dtype conversion and fit only after conversion; the original investigation is retained unchanged.

In [ ]:

import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Preserve your existing adjusted dataset
adjusted_df = adjusted_df.copy()

# Convert categorical variables to compatible types
categorical_cols = [
    "Category_Group",
    "Region_Group",
    "Source_Quarter"
]

for col in categorical_cols:
    adjusted_df[col] = (
        adjusted_df[col]
        .fillna("Unknown")
        .astype(str)
        .astype("object")
    )

# Convert deciles and target to standard numeric types
adjusted_df["Budget_Decile"] = pd.to_numeric(
    adjusted_df["Budget_Decile"],
    errors="coerce"
)

adjusted_df["Target_Failed"] = pd.to_numeric(
    adjusted_df["Target_Failed"],
    errors="coerce"
)

adjusted_df = adjusted_df.dropna(
    subset=["Budget_Decile", "Target_Failed"]
).copy()

adjusted_df["Budget_Decile"] = (
    adjusted_df["Budget_Decile"].astype("int64")
)

adjusted_df["Target_Failed"] = (
    adjusted_df["Target_Failed"].astype("int64")
)

print("DATA TYPES")
print(adjusted_df[
    ["Budget_Decile", "Category_Group",
     "Region_Group", "Source_Quarter",
     "Target_Failed"]
].dtypes)

print("\nObservations:", len(adjusted_df))


### 8.5 Initial adjusted-model attempt (saved error)

This original run computes common 2025 budget boundaries and sets up the categorical interaction model. Its stored output raises `TypeError: Cannot interpret 'string[python]' as a data type`, because the formula interface encounters pandas extension string columns. The successful fit is documented in the following cells; **do not interpret this error-producing cell as a completed result**.

In [ ]:

import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# 2025, Goods and Civil Works only
adjusted_df = df2.loc[
    (df2["Publication_Year"] == 2025) &
    (df2["Category_Group"].isin(
        ["Goods", "Civil Works"]
    ))
].copy()

# Use common budget decile boundaries
# from the full 2025 labeled population
all_2025 = df2.loc[
    df2["Publication_Year"] == 2025
].copy()

_, bins = pd.qcut(
    all_2025["Approved_Budget"],
    q=10,
    retbins=True,
    duplicates="drop"
)

adjusted_df["Budget_Decile"] = pd.cut(
    adjusted_df["Approved_Budget"],
    bins=bins,
    labels=False,
    include_lowest=True
) + 1

# Fit adjusted binomial regression
formula = """
Target_Failed ~
C(Budget_Decile) * C(Category_Group)
+ C(Region_Group)
+ C(Source_Quarter)
"""

adjusted_model = smf.glm(
    formula=formula,
    data=adjusted_df,
    family=sm.families.Binomial()
).fit()

print("Adjusted logistic model fitted.")
print("Observations:", adjusted_model.nobs)

# Estimate category-specific adjusted rates
results = []

for category in ["Civil Works", "Goods"]:

    base = adjusted_df.loc[
        adjusted_df["Category_Group"] == category,
        ["Region_Group", "Source_Quarter"]
    ].copy()

    for decile in range(1, 11):

        scenario = base.copy()
        scenario["Category_Group"] = category
        scenario["Budget_Decile"] = decile

        adjusted_rate = adjusted_model.predict(
            scenario
        ).mean()

        results.append({
            "Category": category,
            "Budget_Decile": decile,
            "Adjusted_Failure_Rate": adjusted_rate
        })

adjusted_results = pd.DataFrame(results)

display(adjusted_results.round(4))

# Plot adjusted relationships
pivot = adjusted_results.pivot(
    index="Budget_Decile",
    columns="Category",
    values="Adjusted_Failure_Rate"
)

ax = (pivot * 100).plot(
    marker="o",
    figsize=(10, 5)
)

ax.set_title(
    "Adjusted Bidding Failure Rates by Budget and Category"
)
ax.set_xlabel("Approved Budget Decile")
ax.set_ylabel("Adjusted Failure Rate (%)")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### 8.6 Fit the adjusted binomial model after dtype preparation

Using the already cleaned `adjusted_df` from the working session, the formula-based model converges on **104,887** observations. This is an **association analysis on the 2025 labeled subset**, not another out-of-time predictive validation and not a causal evaluation.

In [ ]:

formula = (
    "Target_Failed ~ "
    "C(Budget_Decile) * C(Category_Group) "
    "+ C(Region_Group) "
    "+ C(Source_Quarter)"
)

adjusted_model = smf.glm(
    formula=formula,
    data=adjusted_df,
    family=sm.families.Binomial()
).fit()

print("Adjusted Logistic Regression fitted successfully.")
print("Observations:", int(adjusted_model.nobs))
print("Converged:", adjusted_model.converged)


### 8.7 Generate adjusted, category-specific failure curves

To interpret the fitted relationships, the code predicts each budget decile while averaging across the observed region/quarter mix within **Goods** and **Civil Works**. It then plots adjusted failure rates. The saved results indicate that the budget relationship differs materially by category, reinforcing the need for segmentation and cautious interpretation.

In [ ]:

import matplotlib.pyplot as plt

results = []

for category in ["Civil Works", "Goods"]:

    base = adjusted_df.loc[
        adjusted_df["Category_Group"] == category,
        ["Region_Group", "Source_Quarter"]
    ].copy()

    for decile in range(1, 11):

        scenario = base.copy()
        scenario["Category_Group"] = category
        scenario["Budget_Decile"] = decile

        predicted = adjusted_model.predict(scenario)

        results.append({
            "Category": category,
            "Budget_Decile": decile,
            "Adjusted_Failure_Rate_Pct": predicted.mean() * 100
        })

adjusted_results = pd.DataFrame(results)

display(adjusted_results.round(2))

pivot = adjusted_results.pivot(
    index="Budget_Decile",
    columns="Category",
    values="Adjusted_Failure_Rate_Pct"
)

ax = pivot.plot(
    kind="line",
    marker="o",
    figsize=(11, 5)
)

ax.set_title(
    "Adjusted Failure Rate by Budget and Procurement Category"
)

ax.set_xlabel("Approved Budget Decile")
ax.set_ylabel("Adjusted Failure Rate (%)")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Phase 9. Findings, limitations and management implications

### Consolidated results from the saved notebook

| Stage / metric | Recorded result | Interpretation |
|---|---:|---|
| Quarterly source coverage | 14 files, 2023 Q1–2026 Q2 | Later 2026 quarters are not in scope. |
| Consolidated Public Bidding item rows | 1,840,546 | Item/line-level records, **not** unique bids. |
| Unique bid references | 1,055,442 | One reference is the intended modeling unit. |
| Bid-level resolved labels before quality exclusions | 501,471 | Only consistently Awarded or Failed references. |
| Clean final modeling bids | 501,298 | 173 labeled bids excluded for specified feature issues. |
| 2025 validation, original Random Forest | ROC-AUC 0.726; PR-AUC 0.343 | Best tested 2025 ranking model, not a guarantee of operational results. |
| 2025 Random Forest at top 10% | 36.4% precision; 31.7% recall; 3.17× lift | Illustrates the potential value of prioritized review. |
| 2026 original Random Forest | ROC-AUC 0.574; top-10% lift 1.55× | Clear out-of-time degradation. |
| 2026 harmonized Random Forest | ROC-AUC 0.600; top-10% lift 1.57× | Only modest recovery after feature harmonization. |

### What the findings support

The project demonstrates a **defensible exploratory pipeline** and a **measurable 2025 risk-ranking signal**. The strongest immediate use case is a **human-reviewed priority queue**, evaluated under fixed review capacity and compared against the baseline failure rate. The finding is *screening potential*, not confirmed reduction in failed procurements.

### Limitations that must appear in the management presentation

1. **Incomplete and nonrandom outcome labeling:** Many bids remain Closed, Active or otherwise unresolved. Performance is estimated only for bids with usable final labels. This can distort both apparent failure rates and performance across periods.
2. **2026 source/reporting changes:** Procurement mode, funding source and category naming changed. The original model transfers poorly; the simplified harmonized model improves only slightly.
3. **Evaluation is observational:** ROC-AUC, PR-AUC and lift quantify historical ranking on labeled records. They do not demonstrate prevented failures, improved procurement outcomes, or causal effects.
4. **Potential availability / leakage risk:** The intended predictors should be available **at or before notice publication**. Historical tables are used here; the timing and immutability of every field must be verified for prospective deployment.
5. **Representativeness and exclusions:** Only Public Bidding is in scope; other procurement modes and 173 lower-quality labeled observations are not included in model fitting.
6. **No external or real-time validation:** 2026 is a useful stress test but is itself affected by source and label shifts. A forward-verified, stable-schema sample is needed before the model can drive workflow decisions.

### Responsible next step

Keep the notebook's **data audit, label-coverage assessment and time-based validation** as core deliverables. For a management demonstration, show a top-10% review queue and the number of retrospectively observed failures it captures, clearly labeled as a *historical simulation*. Before implementation, verify predictor timing, establish a consistent taxonomy and perform a prospective validation with operational review costs, false-positive workload and subsequent outcomes tracked separately.

**Model-selection position:** Do not continue tuning solely to increase accuracy or AUC. Address label validity, feature comparability and deployment feasibility first.

---
*Documentation was added around the existing analytical cells. Saved calculations, figures and diagnostic errors were retained as recorded; no model was retrained during this documentation pass.*

---
## Continuation on the merged 1.5 + 2.0 dataset

The roadmap above was re-run on the expanded data in three notebooks:

| Notebook | Roadmap steps | Main changes from this run |
|---|---|---|
| `notebooks/01_data_preparation_and_eda.ipynb` | 1–5 | 18 source files (adds PhilGEPS 1.5 for 2026 Q1–Q3 and 2.0 for 2026 Q3). Duplicate 2.0 files for 2023–2025 verified by checksum and skipped. One bid-level table keyed by (source system, bid reference). Field-by-field availability audit. |
| `notebooks/02_modeling.ipynb` | 6 | Random 70/15/15 split of the merged bid table, stratified by outcome and system. The LT8 5-feature set is compared with an expanded at-publication feature set across four models. The selected model is calibrated, so scores read as probabilities. Sensitivity run counting cancellations as failures. |
| `notebooks/03_diagnostics_and_interpretation.ipynb` | 7–9 | Out-of-time check (train 2023–2024, test 2025 and 2026 by system). Drift, label coverage, permutation importance, budget and classification patterns, review-queue simulation. Findings, limitations and answers to the reviewer's questions. |